# CS781 Phase 6 — Fast Models 04–07

Run this notebook on **Kaggle GPU T4 ×2**. It trains four separate abstractive summarizers using your own Phase 1 checkpoints and the official Phase 6 data. It does not rerun Models 01–03 or write to the old Phase 6 project.

| Candidate | Phase 1 parent | What trains |
|---|---|---|
| 04 · Hybrid LoRA | main_hybrid | Original Model 4: rank-16 LoRA on the hybrid Transformer. The recurrent backbone stays frozen. |
| 05 · Decoder soft prompt | main_decoder | A 32-token trainable soft prompt. The decoder weights stay frozen. |
| 06 · Distilled decoder student | main_decoder | Half the original decoder layers, initialized from the Phase 1 decoder. Model 04 teaches next-token probabilities on one in three training microbatches by default. |
| 07 · Cross-attention tuning | main_encoder_decoder | The native decoder's cross-attention projections and associated norms, plus the final norm. The encoder and other decoder weights stay frozen. |

These are four additional **runs**. Keep the already completed 01–03 files for the full comparison, including Model 02 for LoRMA and Model 03 for the local GQA family. Each individual output ZIP contains a single root-level prediction.json. The combined download ZIP is for convenience; upload the four individual ZIPs to the evaluator.

### Why this is faster

- The shared cache selects up to **1,024 source tokens**, retains evidence from the opening and closing of long cases, and reports what fraction of training summaries are clipped by the **640-token target ceiling**. Training batches group similarly sized cases. Decoder and hybrid training concatenate source and target without an internal padding gap.
- The default is **2 epochs, up to 192 update attempts per epoch, and a 20-minute training-loop budget per model**. The limit is checked at update boundaries. Data preparation, dev loss, checkpoints, test generation and optional BERTScore take additional time.
- Checkpoints are selected by teacher-forced loss on a small, fixed dev probe. Greedy KV-cached test generation runs once per model. Distillation scores the gold prefixes with one teacher forward pass when enabled; it does not generate an extra set of teacher summaries.
- Context and update caps can trade off quality. Compare the held-out scores and clipping audit before raising the limits; this notebook does not promise a leaderboard score or measured T4 runtime.


## 1 · Attach inputs and configure

Attach the original **phase-1-outputs** Kaggle dataset. Point PHASE1_ROOT to its direct capstone_phase1 directory containing tokenizer.json and runs/main_{hybrid,decoder,encoder_decoder}/best_model.pt. Attach the official Phase_6_Summ directory with Train/train.json and Test/test.json, or enable Kaggle Internet and add your authorized HF_TOKEN as a Kaggle secret. If the earlier run's official data is still in /kaggle/working/capstone_phase6_v1/data, it is reused automatically.

For a quick trial, set EPOCHS=1 and STEPS_PER_EPOCH=96. For a quality rerun, use a fresh PROJECT and increase the context or update budget only where the audit and held-out results justify it. Keep the settings unchanged when resuming the new notebook. Same-project resume restores completed epochs, so a kernel interruption can replay the current epoch.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

PROJECT = Path('/kaggle/working/capstone_phase6_fast')
PHASE1_ROOT = Path('/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1')
DATA_ROOT = None  # Or Path('/kaggle/input/.../Phase_6_Summ')
DATA_REVISION = None  # Optional official HF dataset commit when downloading
WORLD = 2

RUN_MODELS = [
    '04_hybrid_lora',
    '05_decoder_prompt',
    '06_decoder_student_kd',
    '07_encoder_decoder_cross_attention',
]
# To run only 04, keep just its name. Selecting 06 automatically trains/reuses 04 as teacher.
SOURCE_TOKENS = 1024
TARGET_TOKENS = 640
TRAIN_POOL = 6144       # 0 = preprocess all training documents
EPOCHS = 2
STEPS_PER_EPOCH = 192
MAX_TRAIN_MINUTES = 20  # Training loop only; 0 disables the clock limit
MICRO_BATCH = 2         # Per GPU; set to 1 and ACCUMULATION=8 if GPU memory is tight
ACCUMULATION = 4
GENERATION_BATCH = 4   # Automatically falls back to single-document decoding on OOM
CHECKPOINTING = False  # Enable if needed after reducing the training microbatch
EVAL_DEV_CASES = 24     # 0 skips optional BERTScore/ROUGE; test ZIPs are generated first

# Optional warm start from the SLOW notebook's completed 04 epoch checkpoint.
# Supply BOTH paths, e.g. .../runs/04_hybrid_lora/best.pt and .../token_cache/cache_v4.pt.
# The parent, tokenizer and old split are checked. This starts a fresh optimizer and budget.
OLD_MODEL_04_WEIGHTS = None
OLD_MODEL_04_CACHE = None

if not 128 <= SOURCE_TOKENS or TARGET_TOKENS < 32:
    raise ValueError('Use SOURCE_TOKENS >= 128 and TARGET_TOKENS >= 32.')
if min(EPOCHS, STEPS_PER_EPOCH, MICRO_BATCH, ACCUMULATION, GENERATION_BATCH) < 1:
    raise ValueError('Training and batching settings must be positive.')
if min(TRAIN_POOL, MAX_TRAIN_MINUTES, EVAL_DEV_CASES) < 0:
    raise ValueError('Pool, clock and metric settings cannot be negative.')
if bool(OLD_MODEL_04_WEIGHTS) != bool(OLD_MODEL_04_CACHE):
    raise ValueError('Set both warm-start paths, or leave both as None.')

required = {'tokenizers':'tokenizers', 'sklearn':'scikit-learn', 'huggingface_hub':'huggingface_hub'}
missing = [package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import torch
if torch.cuda.device_count() < WORLD:
    raise RuntimeError('Select GPU T4 ×2 under Kaggle Accelerator and restart the session.')
if not hasattr(torch.nn.utils,'get_total_norm'):
    raise RuntimeError('Use a Kaggle runtime with PyTorch 2.6 or newer.')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
PROJECT.mkdir(parents=True, exist_ok=True)
CODE = PROJECT / 'code'
CODE.mkdir(exist_ok=True)
if str(CODE) not in sys.path:
    sys.path.insert(0,str(CODE))
for rank in range(WORLD):
    print(f'GPU {rank}: {torch.cuda.get_device_name(rank)}')
print('Effective batch:', WORLD * MICRO_BATCH * ACCUMULATION, 'documents per optimizer update.')
print('Output project:', PROJECT)


## 2 · Write the self-contained runtime

Run all the cells in this section. They embed the data parser, original Phase 1 model definitions, and the fast training/export implementation. No other notebook or code attachment is needed.


In [ ]:
# legal_models.py
module_source = '"""Four randomly initialized legal language models; no downloaded model weights."""\nimport math\nfrom dataclasses import dataclass, asdict\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\n@dataclass\nclass ModelConfig:\n    architecture: str = "decoder"\n    vocab_size: int = 32768\n    dim: int = 512\n    heads: int = 8\n    kv_heads: int = 8\n    layers: int = 8\n    ff_dim: int = 2048\n    encoder_layers: int = 0\n    rnn_layers: int = 0\n    local_window: int = 0\n    max_positions: int = 4096\n    dropout: float = 0.0\n    gradient_checkpointing: bool = False\n\n\ndef architecture_config(name, **overrides):\n    c = ModelConfig(architecture=name)\n    if name == "decoder":\n        pass\n    elif name == "local_gqa":\n        c.kv_heads, c.layers, c.local_window = 2, 9, 256\n    elif name == "hybrid":\n        c.layers, c.rnn_layers = 7, 2\n    elif name == "encoder_decoder":\n        c.layers, c.encoder_layers = 4, 3\n    else:\n        raise ValueError(f"Unknown architecture: {name}")\n    for k, v in overrides.items():\n        if not hasattr(c, k):\n            raise ValueError(k)\n        setattr(c, k, v)\n    assert c.dim % c.heads == 0 and c.heads % c.kv_heads == 0\n    assert (c.dim // c.heads) % 2 == 0\n    return c\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-6):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        # Reduction in FP32 is useful with T4 FP16 training.\n        y = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)\n        return y.to(x.dtype) * self.weight.to(x.dtype)\n\n\nclass Rotary(nn.Module):\n    def __init__(self, head_dim, max_positions):\n        super().__init__()\n        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))\n        angles = torch.outer(torch.arange(max_positions).float(), inv)\n        self.register_buffer("cos", angles.cos(), persistent=False)\n        self.register_buffer("sin", angles.sin(), persistent=False)\n\n    def forward(self, x, offset=0):\n        n = x.size(-2)\n        if offset + n > self.cos.size(0):\n            raise ValueError("Position limit exceeded; increase max_positions in the model config.")\n        cos = self.cos[offset:offset + n].to(x.dtype)[None, None]\n        sin = self.sin[offset:offset + n].to(x.dtype)[None, None]\n        a, b = x[..., 0::2], x[..., 1::2]\n        return torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1).flatten(-2)\n\n\nclass Attention(nn.Module):\n    def __init__(self, c, causal=True, cross=False):\n        super().__init__()\n        self.heads, self.kv_heads = c.heads, c.kv_heads\n        self.hd = c.dim // c.heads\n        self.q = nn.Linear(c.dim, c.dim, bias=False)\n        self.k = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)\n        self.v = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)\n        self.out = nn.Linear(c.dim, c.dim, bias=False)\n        self.rotary = None if cross else Rotary(self.hd, c.max_positions)\n        self.causal, self.cross = causal, cross\n        self.window = c.local_window if causal and not cross else 0\n        self.dropout = c.dropout\n\n    def forward(self, x, memory=None, past=None, position=0, use_cache=False):\n        b, t, _ = x.shape\n        q = self.q(x).view(b, t, self.heads, self.hd).transpose(1, 2)\n        if self.cross and past is not None:\n            k, v = past  # Encoder K/V projections are also cached during decoding.\n        else:\n            source = x if memory is None else memory\n            s = source.size(1)\n            k = self.k(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)\n            v = self.v(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)\n            if self.rotary is not None:\n                k = self.rotary(k, position)\n            if past is not None:\n                k, v = torch.cat((past[0], k), -2), torch.cat((past[1], v), -2)\n        if self.rotary is not None:\n            q = self.rotary(q, position)\n        new_cache = (k, v) if use_cache else None\n        # Portable GQA: project/cache fewer KV heads, repeat only for the SDPA call.\n        # This is not a block-sparse implementation or a guaranteed training speedup.\n        if self.kv_heads != self.heads:\n            repeats = self.heads // self.kv_heads\n            k = k.repeat_interleave(repeats, dim=1)\n            v = v.repeat_interleave(repeats, dim=1)\n        mask, is_causal = None, False\n        if self.causal:\n            s = k.size(-2)\n            if self.window or (past is not None and t > 1):\n                qi = torch.arange(s - t, s, device=x.device)[:, None]\n                ki = torch.arange(s, device=x.device)[None, :]\n                mask = ki <= qi\n                if self.window:\n                    mask = mask & ((qi - ki) < self.window)\n            elif past is None:\n                is_causal = True\n            # Cached single-token query must see all prior keys. is_causal=True\n            # here would incorrectly use the upper-left non-square mask.\n        y = F.scaled_dot_product_attention(\n            q, k, v, attn_mask=mask, is_causal=is_causal,\n            dropout_p=self.dropout if self.training else 0.0,\n        )\n        y = self.out(y.transpose(1, 2).contiguous().view(b, t, -1))\n        return y, new_cache\n\n\nclass SwiGLU(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.gate = nn.Linear(c.dim, c.ff_dim, bias=False)\n        self.up = nn.Linear(c.dim, c.ff_dim, bias=False)\n        self.down = nn.Linear(c.ff_dim, c.dim, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\n\nclass Block(nn.Module):\n    def __init__(self, c, encoder=False):\n        super().__init__()\n        self.attn_norm, self.ff_norm = RMSNorm(c.dim), RMSNorm(c.dim)\n        self.attn = Attention(c, causal=not encoder)\n        self.ff = SwiGLU(c)\n        self.drop = nn.Dropout(c.dropout)\n\n    def forward(self, x, past=None, position=0, use_cache=False):\n        y, cache = self.attn(self.attn_norm(x), past=past, position=position, use_cache=use_cache)\n        x = x + self.drop(y)\n        x = x + self.drop(self.ff(self.ff_norm(x)))\n        return (x, cache) if use_cache else x\n\n\nclass CrossBlock(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.self_norm, self.cross_norm, self.ff_norm = (RMSNorm(c.dim) for _ in range(3))\n        self.self_attn = Attention(c)\n        self.cross_attn = Attention(c, causal=False, cross=True)\n        self.ff, self.drop = SwiGLU(c), nn.Dropout(c.dropout)\n\n    def forward(self, x, memory, past=None, cross_past=None, position=0, use_cache=False):\n        y, cache = self.self_attn(self.self_norm(x), past=past, position=position, use_cache=use_cache)\n        x = x + self.drop(y)\n        y, cross_cache = self.cross_attn(self.cross_norm(x), memory=memory, past=cross_past, use_cache=use_cache)\n        x = x + self.drop(y)\n        x = x + self.drop(self.ff(self.ff_norm(x)))\n        return (x, cache, cross_cache) if use_cache else x\n\n\nclass LegalLM(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.config = c\n        self.embedding = nn.Embedding(c.vocab_size, c.dim)\n        # Output projection uses embedding.weight; there is no second vocabulary matrix.\n        if c.architecture == "encoder_decoder":\n            self.encoder = nn.ModuleList(Block(c, encoder=True) for _ in range(c.encoder_layers))\n            self.encoder_norm = RMSNorm(c.dim)\n            self.blocks = nn.ModuleList(CrossBlock(c) for _ in range(c.layers))\n        else:\n            self.blocks = nn.ModuleList(Block(c) for _ in range(c.layers))\n        if c.rnn_layers:\n            self.rnn = nn.LSTM(c.dim, c.dim, num_layers=c.rnn_layers, batch_first=True)\n            self.rnn_norm = RMSNorm(c.dim)\n        self.final_norm = RMSNorm(c.dim)\n        self.apply(self._init)\n        # Scale residual projections to keep initial residual variance controlled.\n        std = 0.02 / math.sqrt(2 * c.layers)\n        for name, p in self.named_parameters():\n            if name.endswith(("attn.out.weight", "self_attn.out.weight", "cross_attn.out.weight", "ff.down.weight")):\n                nn.init.normal_(p, mean=0.0, std=std)\n\n    @staticmethod\n    def _init(m):\n        if isinstance(m, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(m.weight, mean=0.0, std=0.02)\n        elif isinstance(m, nn.LSTM):\n            for name, p in m.named_parameters():\n                if "weight" in name:\n                    nn.init.normal_(p, mean=0.0, std=0.02)\n                else:\n                    nn.init.zeros_(p)\n\n    def encode(self, source_ids):\n        x = self.embedding(source_ids)\n        for block in self.encoder:\n            if self.config.gradient_checkpointing and self.training:\n                x = checkpoint(block, x, use_reentrant=False)\n            else:\n                x = block(x)\n        return self.encoder_norm(x)\n\n    def forward(self, input_ids, source_ids=None, state=None, use_cache=False):\n        state = state or {}\n        position = state.get("position", 0)\n        x = self.embedding(input_ids)\n        new_state = {"position": position + input_ids.size(1), "self": []}\n        if self.config.rnn_layers:\n            y, recurrent_state = self.rnn(x, state.get("rnn"))\n            x = x + self.rnn_norm(y)\n            new_state["rnn"] = recurrent_state\n        if self.config.architecture == "encoder_decoder":\n            memory = state.get("memory")\n            if memory is None:\n                if source_ids is None:\n                    raise ValueError("encoder_decoder requires source_ids on the first call.")\n                memory = self.encode(source_ids)\n            new_state.update(memory=memory, cross=[])\n            for i, block in enumerate(self.blocks):\n                if use_cache:\n                    p = state.get("self", [None] * len(self.blocks))[i]\n                    cp = state.get("cross", [None] * len(self.blocks))[i]\n                    x, cache, cc = block(x, memory, p, cp, position, True)\n                    new_state["self"].append(cache)\n                    new_state["cross"].append(cc)\n                elif self.config.gradient_checkpointing and self.training:\n                    x = checkpoint(block, x, memory, use_reentrant=False)\n                else:\n                    x = block(x, memory)\n        else:\n            for i, block in enumerate(self.blocks):\n                if use_cache:\n                    p = state.get("self", [None] * len(self.blocks))[i]\n                    x, cache = block(x, p, position, True)\n                    new_state["self"].append(cache)\n                elif self.config.gradient_checkpointing and self.training:\n                    x = checkpoint(block, x, use_reentrant=False)\n                else:\n                    x = block(x)\n        x = self.final_norm(x)\n        return (x, new_state) if use_cache else x\n\n    def project(self, hidden):\n        return F.linear(hidden, self.embedding.weight)\n\n\ndef parameter_count(model):\n    # PyTorch de-duplicates shared Parameter objects here.\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef parameter_formula(c):\n    d, h, kv = c.dim, c.dim // c.heads, c.kv_heads\n    attention = 2 * d * d + 2 * d * kv * h\n    ff = 3 * d * c.ff_dim\n    total = c.vocab_size * d + d\n    if c.architecture == "encoder_decoder":\n        total += c.encoder_layers * (attention + ff + 2 * d) + d\n        total += c.layers * (2 * attention + ff + 3 * d)\n    else:\n        total += c.layers * (attention + ff + 2 * d)\n    if c.rnn_layers:\n        total += c.rnn_layers * (8 * d * d + 8 * d) + d\n    return total\n\n\ndef make_batch(blocks, architecture, prefix_fraction=0.80, objective="full"):\n    # blocks has T+1 tokens. The target suffix is identical across architectures.\n    t = blocks.size(1) - 1\n    split = max(1, min(t, int(t * prefix_fraction)))\n    if architecture == "encoder_decoder":\n        source = blocks[:, :split]\n        inputs, targets = blocks[:, split - 1:-1], blocks[:, split:]\n    else:\n        source = None\n        inputs, targets = blocks[:, :-1], blocks[:, 1:].clone()\n        if objective == "continuation":\n            targets[:, :split - 1] = -100\n    return inputs, targets, source\n\n\ndef language_model_loss(hidden, targets, weight, chunk_tokens=256, checkpoint_chunks=False):\n    """Sum/token count lets validation aggregate the true token-weighted NLL."""\n    hidden, targets = hidden.reshape(-1, hidden.size(-1)), targets.reshape(-1)\n    total = hidden.new_zeros((), dtype=torch.float32)\n    count = (targets != -100).sum()\n    def loss_chunk(h, y, w):\n        # Keep logits in autocast precision until the bounded FP32 CE conversion.\n        return F.cross_entropy(F.linear(h, w).float(), y, ignore_index=-100, reduction="sum")\n    for i in range(0, targets.numel(), chunk_tokens):\n        h, y = hidden[i:i + chunk_tokens], targets[i:i + chunk_tokens]\n        if checkpoint_chunks and torch.is_grad_enabled():\n            total = total + checkpoint(loss_chunk, h, y, weight, use_reentrant=False)\n        else:\n            total = total + loss_chunk(h, y, weight)\n    return total / count.clamp_min(1), total.detach(), count.detach()\n\n\nif __name__ == "__main__":\n    for name in ("decoder", "encoder_decoder", "local_gqa", "hybrid"):\n        c = architecture_config(name)\n        with torch.device("meta"):\n            model = LegalLM(c)\n        n = parameter_count(model)\n        assert n == parameter_formula(c) and 50_000_000 <= n <= 300_000_000\n        print(name, f"{n:,}", asdict(c))\n'
(CODE / 'legal_models.py').write_text(module_source, encoding="utf-8")
print("Wrote legal_models.py")


In [ ]:
# s6_data.py
module_source = '"""Official Phase-6 data, train-only salience selector, auditable cached contexts."""\nimport hashlib,json,re,os,pickle,collections\nfrom pathlib import Path\nimport numpy as np\nfrom tokenizers import Tokenizer\n\nFILES=[\'Train/train.json\',\'Test/test.json\']\ndef digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()\ndef file_hash(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(1<<20),b\'\'):h.update(b)\n    return h.hexdigest()\ndef read_json(p):\n    def unique(pairs):\n        d={}\n        for k,v in pairs:\n            if k in d:raise ValueError(f\'Duplicate JSON mapping key: {k}\')\n            d[k]=v\n        return d\n    return json.loads(Path(p).read_text(encoding=\'utf-8-sig\'),object_pairs_hook=unique)\ndef atomic_json(p,x):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    t=p.with_name(p.name+\'.tmp\');t.write_text(json.dumps(x,ensure_ascii=False,indent=2));t.replace(p)\ndef alias(r,names):\n    keys=[k for k in names if k in r]\n    if len(keys)>1 and any(r[k]!=r[keys[0]] for k in keys):raise ValueError(f\'Conflicting fields: {keys}\')\n    return r[keys[0]] if keys else None\nDOCUMENT_FIELDS=(\'document\',\'text\',\'sentences\')\nSUMMARY_FIELDS=(\'summary\',\'summaries\',\'abstract\',\'reference_summary\',\'reference_summaries\',\'gold_summary\',\'summary_text\',\'abstractive_summary\',\'reference\',\'highlights\',\'target\')\ndef text_value(v,fields,depth=0):\n    """Normalize text and explicit text containers, never stringify arbitrary metadata."""\n    if depth>5:return None\n    if isinstance(v,str):return v.strip() or None\n    if isinstance(v,list):\n        if all(isinstance(s,str) or s is None for s in v):\n            return \'\\n\'.join(s.strip() for s in v if isinstance(s,str) and s.strip()) or None\n        # A list of structured references can contain alternatives; use the first usable one.\n        for item in v:\n            result=text_value(item,fields,depth+1)\n            if result:return result\n    elif isinstance(v,dict):\n        for key in fields:\n            if key in v:\n                result=text_value(v[key],fields,depth+1)\n                if result:return result\n    return None\ndef records(v,split,audit=None):\n    if isinstance(v,dict) and len(v)==1 and next(iter(v)) in (\'data\',\'records\',\'documents\'):v=next(iter(v.values()))\n    if isinstance(v,list):items=[(None,r) for r in v]\n    elif isinstance(v,dict):\n        ids=alias(v,(\'id\',\'ids\',\'doc_id\',\'case_id\'));texts=alias(v,DOCUMENT_FIELDS)\n        summary_columns={} if split==\'test\' else {k:v[k] for k in SUMMARY_FIELDS if k in v}\n        if isinstance(ids,str) and texts is not None:items=[(None,v)]\n        elif isinstance(ids,list) and isinstance(texts,list):\n            if len(ids)!=len(texts) or any(v is not None and (not isinstance(v,list) or len(v)!=len(ids)) for v in summary_columns.values()):raise ValueError(f\'{split}: column lengths differ; cannot align documents and summaries by position.\')\n            items=[(None,{\'id\':k,\'document\':texts[i],**{f:col[i] for f,col in summary_columns.items() if col is not None}}) for i,k in enumerate(ids)]\n        elif isinstance(texts,dict) and ids is None:\n            if any(col is not None and (not isinstance(col,dict) or set(col)-set(texts)) for col in summary_columns.values()):raise ValueError(f\'{split}: summary columns contain unexpected IDs or are not mappings.\')\n            items=[(None,{\'id\':k,\'document\':t,**{f:col.get(k) for f,col in summary_columns.items() if col is not None}}) for k,t in texts.items()]\n        else:items=list(v.items())\n    else:raise ValueError(\'Expected list of records or ID mapping.\')\n    out={};duplicates=0;skipped=[];fields=collections.Counter()\n    for key,r in items:\n        if isinstance(r,(str,list)) and key is not None:r={\'id\':key,\'document\':r}\n        if not isinstance(r,dict):raise ValueError(f\'{split}: record must be an object; found {type(r).__name__}.\')\n        did=alias(r,(\'id\',\'doc_id\',\'case_id\'));did=key if did is None else did\n        if not isinstance(did,str) or not did.strip():raise ValueError(\'Original nonempty string IDs required; IDs are never invented.\')\n        if key is not None and key!=did:raise ValueError(\'Mapping key and explicit ID differ.\')\n        text=text_value(alias(r,DOCUMENT_FIELDS),DOCUMENT_FIELDS+(\'content\',\'sentence\'))\n        # Deliberately do not even inspect supplied test reference fields.\n        summary=None\n        if split!=\'test\':\n            for field in SUMMARY_FIELDS:\n                if field not in r:continue\n                summary=text_value(r[field],SUMMARY_FIELDS+(\'text\',\'content\',\'sentences\',\'sentence\'))\n                if summary:\n                    fields[field]+=1;break\n        if not text or (split!=\'test\' and not summary):\n            reason=\'missing or unusable document\' if not text else \'missing or unusable reference summary\'\n            if split==\'test\':raise ValueError(f\'test ID {did!r}: {reason}; cannot silently drop an official test ID.\')\n            skipped.append({\'id\':did,\'reason\':reason,\'field_types\':{k:type(val).__name__ for k,val in r.items()}})\n            continue\n        row={\'id\':did,\'document\':text,\'summary\':summary,\'split\':split}\n        if did in out:\n            if row!=out[did]:raise ValueError(f\'{split}: conflicting duplicate ID {did}; inspect source instead of inventing an ID.\')\n            duplicates+=1;continue\n        out[did]=row\n    if audit is not None:audit.update(input_rows=len(items),usable_rows=len(out),skipped_rows=skipped,reference_fields_used=dict(fields))\n    if not out:raise ValueError(f\'No usable {split} records. Examples (IDs and field types only): {skipped[:5]}\')\n    return list(out.values()),duplicates\ndef normhash(text):return digest(\' \'.join(text.lower().split()))\ndef split_data(root,seed=781,dev_fraction=.08,dev_max=512):\n    train_audit={};test_audit={}\n    train,duptr=records(read_json(Path(root)/FILES[0]),\'train\',train_audit);test,dupte=records(read_json(Path(root)/FILES[1]),\'test\',test_audit)\n    forbidden_ids={r[\'id\'] for r in test};forbidden_text={normhash(r[\'document\']) for r in test}\n    removed=[r[\'id\'] for r in train if r[\'id\'] in forbidden_ids or normhash(r[\'document\']) in forbidden_text]\n    removed_ids=set(removed);train=[r for r in train if r[\'id\'] not in removed_ids]\n    # Group exact normalized document copies before splitting; retain one training reference.\n    groups=collections.defaultdict(list)\n    for r in train:groups[normhash(r[\'document\'])].append(r)\n    keys=sorted(groups);np.random.default_rng(seed).shuffle(keys)\n    if len(keys)<4:raise ValueError(f\'Need at least four distinct labeled train documents after cleanup; got {len(keys)}. Skipped rows (examples): {train_audit["skipped_rows"][:5]}\')\n    n=max(1,min(dev_max,round(len(keys)*dev_fraction),len(keys)-2));held=set(keys[:n])\n    rows=[];conflicting=0\n    for key in keys:\n        group=groups[key];conflicting+=len({r[\'summary\'] for r in group})>1\n        r=dict(sorted(group,key=lambda x:x[\'id\'])[0]);r[\'split\']=\'dev\' if key in held else \'train\';rows.append(r)\n    rows+=test\n    audit={\'duplicate_id_copies_removed\':{\'train\':duptr,\'test\':dupte},\'train_test_overlap_removed\':removed,\n           \'train_records\':train_audit,\'test_records\':test_audit,\n           \'normalized_document_copies_collapsed\':sum(len(v)-1 for v in groups.values()),\n           \'copy_groups_with_multiple_references\':conflicting,\'dev_group_count\':n,\n           \'test_references\':\'discarded without inspection; never cached\',\'seed\':seed}\n    return rows,audit\ndef sentences(text):\n    # Deterministic, no NLTK resource downloads. Keep sentence boundaries and source order.\n    parts=[s.strip() for s in re.split(r\'(?<=[.!?])\\s+(?=[A-Z0-9(])|\\n+\',text) if s.strip()]\n    # Cap individual fragments for bounded selector features; no head-only document truncation.\n    return [s[i:i+1800] for s in parts for i in range(0,len(s),1800)] or [text]\ndef wordset(s):return set(re.findall(r\'[a-z0-9]+\',s.lower()))\nCUE=re.compile(r\'\\b(held|hold|therefore|dismissed|allowed|appeal|issue|court|section|petitioner|respondent|conclusion|ordered|judgment)\\b\',re.I)\nclass Selector:\n    """Hashing salience regression + position/cue features; fitted on training references only."""\n    def __init__(self,seed=781):\n        from sklearn.feature_extraction.text import HashingVectorizer\n        from sklearn.linear_model import SGDRegressor\n        self.vectorizer=HashingVectorizer(n_features=2**18,ngram_range=(1,2),alternate_sign=False,norm=\'l2\')\n        self.model=SGDRegressor(loss=\'huber\',epsilon=.15,alpha=1e-5,learning_rate=\'invscaling\',eta0=.05,random_state=seed)\n    def features(self,ss):\n        from scipy.sparse import hstack,csr_matrix\n        n=len(ss);extra=np.asarray([[i/max(1,n-1),float(i<3),float(i>=n-3),min(5,len(CUE.findall(s)))/5,min(1,len(s)/600)] for i,s in enumerate(ss)],dtype=np.float32)\n        return hstack([self.vectorizer.transform(ss),csr_matrix(extra)],format=\'csr\')\n    def fit(self,rows,passes=2):\n        for _ in range(passes):\n            for r in rows:\n                ss=sentences(r[\'document\']);ref=wordset(r[\'summary\']);y=[]\n                for s in ss:\n                    w=wordset(s);overlap=len(w&ref)\n                    # Sentence-level lexical F score is a cheap salience training signal, not the evaluation metric.\n                    y.append(2*overlap/max(1,len(w)+min(len(ref),80)))\n                self.model.partial_fit(self.features(ss),np.asarray(y,dtype=np.float32))\n        return self\n    def select(self,text,tok,budget):\n        ss=sentences(text);encoded=tok.encode_batch(ss,add_special_tokens=False);ids=[e.ids for e in encoded]\n        if sum(map(len,ids))+len(ss)<=budget:return tok.encode(\'\\n\'.join(ss),add_special_tokens=False).ids[:budget],len(ss),len(ss)\n        scores=self.model.predict(self.features(ss));sets=list(map(wordset,ss));selected=[];used=0;remaining=set(range(len(ss)))\n        # Reserve a little space for facts and final disposition; fill most of the context by learned salience + MMR.\n        for i in list(dict.fromkeys([0,len(ss)-1])):\n            take=min(len(ids[i]),max(1,budget//12))\n            if take:selected.append((i,ids[i][:take]));used+=take+1;remaining.discard(i)\n        while remaining and used<budget-8:\n            def score(i):\n                redundancy=max((len(sets[i]&sets[j])/max(1,len(sets[i]|sets[j])) for j,_ in selected),default=0)\n                return float(scores[i])-.15*redundancy\n            i=max(remaining,key=lambda j:(score(j),-j));remaining.remove(i)\n            room=budget-used-1\n            if len(ids[i])>room and selected:continue\n            selected.append((i,ids[i][:room]));used+=min(room,len(ids[i]))+1\n        sep=tok.encode(\'\\n\',add_special_tokens=False).ids\n        result=[]\n        for _,tokens in sorted(selected):result+=tokens+sep\n        return result[:budget],len(selected),len(ss)\ndef special_ids(tok,overrides=None):\n    overrides=overrides or {};out={}\n    for name,candidates in {\'pad\':[\'<pad>\',\'[PAD]\'],\'bos\':[\'<bos>\',\'[BOS]\',\'<s>\'],\'eos\':[\'<eos>\',\'[EOS]\',\'</s>\']}.items():\n        value=overrides.get(name)\n        if value is None:value=next((tok.token_to_id(s) for s in candidates if tok.token_to_id(s) is not None),None)\n        if value is None:raise ValueError(f\'Cannot identify {name} token. Set SPECIAL_IDS explicitly from tokenizer vocabulary.\')\n        if not 0<=int(value)<tok.get_vocab_size():raise ValueError(\'Special ID outside tokenizer vocabulary.\')\n        out[name]=int(value)\n    if len(set(out.values()))!=3:raise ValueError(\'PAD, BOS and EOS must be distinct existing tokens.\')\n    return out\n'
(CODE / 's6_data.py').write_text(module_source, encoding="utf-8")
print("Wrote s6_data.py")


In [ ]:
# s6_model.py
module_source = '"""Original Phase-1 weights with conditional generation, PEFT and KV caches."""\nimport math,types,contextlib\nfrom dataclasses import fields\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\nfrom legal_models import LegalLM,ModelConfig,Attention\nfrom s6_data import file_hash\n\nclass LowRank(nn.Module):\n    def __init__(self,base,rank,alpha,multiplicative=False):\n        super().__init__();self.base=base;self.scale=alpha/rank;self.multiplicative=multiplicative;base.requires_grad_(False)\n        self.A=nn.Linear(base.out_features if multiplicative else base.in_features,rank,bias=False)\n        self.B=nn.Linear(rank,base.out_features,bias=False)\n        nn.init.normal_(self.A.weight,std=.02);nn.init.zeros_(self.B.weight)\n    def forward(self,x):\n        y=self.base(x)\n        # LoRMA+ Eq.8: (I + alpha/r BA) W x. LoRA: W x + alpha/r BA x.\n        return y+self.scale*self.B(self.A(y if self.multiplicative else x))\n@contextlib.contextmanager\ndef merged_adapters(model):\n    """Exact FP32 weight algebra for LoRA/LoRMA+: remove adapter matmul overhead while decoding."""\n    replacements=[]\n    with torch.no_grad():\n        for name,m in list(model.named_modules()):\n            if not isinstance(m,LowRank):continue\n            parent_name,attr=name.rsplit(\'.\',1);parent=model.get_submodule(parent_name)\n            layer=nn.Linear(m.base.in_features,m.base.out_features,bias=m.base.bias is not None,device=m.base.weight.device,dtype=m.base.weight.dtype)\n            update=m.B.weight@(m.A.weight@m.base.weight if m.multiplicative else m.A.weight)\n            layer.weight.copy_(m.base.weight+m.scale*update)\n            if m.base.bias is not None:\n                bias=m.base.bias\n                layer.bias.copy_(bias+m.scale*m.B.weight@(m.A.weight@bias) if m.multiplicative else bias)\n            layer.requires_grad_(False);setattr(parent,attr,layer);replacements.append((parent,attr,m))\n    try:yield\n    finally:\n        for parent,attr,original in replacements:setattr(parent,attr,original)\nclass Adapter(nn.Module):\n    def __init__(self,d,b):\n        super().__init__();self.norm=nn.LayerNorm(d);self.down=nn.Linear(d,b);self.up=nn.Linear(b,d)\n        nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)\n    def forward(self,x):return x+self.up(F.gelu(self.down(self.norm(x))))\ndef attention(self,x,memory=None,past=None,position=0,use_cache=False):\n    b,t,_=x.shape;q=self.q(x).view(b,t,self.heads,self.hd).transpose(1,2)\n    if self.cross and past is not None:k,v=past\n    else:\n        src=x if memory is None else memory;s=src.size(1)\n        k=self.k(src).view(b,s,self.kv_heads,self.hd).transpose(1,2);v=self.v(src).view(b,s,self.kv_heads,self.hd).transpose(1,2)\n        if self.rotary is not None:k=self.rotary(k,position)\n        if past is not None:k=torch.cat([past[0],k],-2);v=torch.cat([past[1],v],-2)\n    if self.rotary is not None:q=self.rotary(q,position)\n    cache=(k,v) if use_cache else None\n    if self.heads!=self.kv_heads:k=k.repeat_interleave(self.heads//self.kv_heads,1);v=v.repeat_interleave(self.heads//self.kv_heads,1)\n    if self._valid is None:\n        if past is not None or use_cache or self.window:\n            raise ValueError(\'Implicit causal mask requires dense right-padded training input.\')\n        y=F.scaled_dot_product_attention(q,k,v,is_causal=self.causal,dropout_p=self.dropout if self.training else 0.)\n        return self.out(y.transpose(1,2).contiguous().view(b,t,-1)),cache\n    s=k.shape[-2];valid=self._valid[:,:s];allowed=valid[:,None,None,:].expand(b,1,t,s)\n    if self.causal:\n        qi=torch.arange(position,position+t,device=x.device)[:,None];ki=torch.arange(s,device=x.device)[None,:]\n        relation=ki<=qi\n        if self.window:relation=relation&((qi-ki)<self.window)\n        allowed=allowed&relation[None,None]\n    # Every real query sees at least BOS/source; padded queries may be fully masked.\n    # SDPA handles empty query rows as zero. No FP16 -1e9 masked-fill path.\n    y=F.scaled_dot_product_attention(q,k,v,attn_mask=allowed,dropout_p=self.dropout if self.training else 0.)\n    return self.out(y.transpose(1,2).contiguous().view(b,t,-1)),cache\ndef load_parent(path,tokenizer_sha,allow_tiny=False):\n    saved=torch.load(path,map_location=\'cpu\',weights_only=False)\n    if saved.get(\'training_origin\')!=\'random_initialization_official_phase1_only\':raise ValueError(\'Parent must be an original own Phase-1 checkpoint, not an adapted task checkpoint.\')\n    if saved.get(\'tokenizer_sha256\')!=tokenizer_sha:raise ValueError(\'Parent/tokenizer SHA256 mismatch.\')\n    c=ModelConfig(**{k:v for k,v in saved[\'model_config\'].items() if k in {f.name for f in fields(ModelConfig)}})\n    c.gradient_checkpointing=False;model=LegalLM(c);model.load_state_dict(saved[\'model\'],strict=True)\n    n=sum(p.numel() for p in model.parameters())\n    if not allow_tiny and not 50_000_000<=n<=300_000_000:raise ValueError(f\'Original parent has {n:,} parameters, outside 50M–300M.\')\n    return model\nclass Summarizer(nn.Module):\n    def __init__(self,backbone,cfg):\n        super().__init__();self.backbone=backbone;self.cfg=cfg;self.native=backbone.config.architecture==\'encoder_decoder\'\n        method=cfg[\'method\'];backbone.requires_grad_(method in (\'full\',\'kd\'))\n        if method==\'kd\':\n            count=max(1,len(backbone.blocks)//2);indices=np.linspace(0,len(backbone.blocks)-1,count).round().astype(int).tolist()\n            backbone.blocks=nn.ModuleList([backbone.blocks[i] for i in indices]);backbone.config.layers=count;self.student_layers=indices\n        if method in (\'lora\',\'lorma\'):\n            targets={\'q\',\'k\',\'v\',\'out\',\'gate\',\'up\',\'down\'}\n            replacements=[(name,m) for name,m in backbone.named_modules() if isinstance(m,nn.Linear) and name.rsplit(\'.\',1)[-1] in targets]\n            for name,m in replacements:\n                parent,attr=name.rsplit(\'.\',1);setattr(backbone.get_submodule(parent),attr,LowRank(m,cfg.get(\'rank\',16),cfg.get(\'alpha\',32),method==\'lorma\'))\n        if method==\'adapter\':\n            for block in backbone.blocks:\n                block.after_attention_adapter=Adapter(backbone.config.dim,cfg.get(\'adapter_dim\',64))\n                block.after_ff_adapter=Adapter(backbone.config.dim,cfg.get(\'adapter_dim\',64))\n        if method==\'cross_only\':\n            if not self.native:raise ValueError(\'Cross-attention tuning needs a native encoder-decoder.\')\n            for block in backbone.blocks:\n                block.cross_attn.requires_grad_(True)\n                block.cross_norm.requires_grad_(True)\n            backbone.final_norm.requires_grad_(True)\n        if method==\'prompt\':\n            n=cfg.get(\'prompt_tokens\',32)\n            self.prompt=nn.Parameter(backbone.embedding.weight.detach()[:n].clone())\n        for m in backbone.modules():\n            if isinstance(m,Attention):\n                m.forward=types.MethodType(attention,m);m.dropout=cfg.get(\'dropout\',.05)\n                # Keep GQA projections; allow long-document information instead of losing source outside a 256-token window.\n                if cfg.get(\'globalize_local\',False):m.window=0\n            if isinstance(m,nn.Dropout):m.p=cfg.get(\'dropout\',.05)\n        self.checkpointing=cfg.get(\'checkpointing\',True)\n    def block(self,block,x,valid,memory=None,source_valid=None,position=0,past=None,cross_past=None,cache=False):\n        if self.native and memory is not None:\n            block.self_attn._valid=valid;block.cross_attn._valid=source_valid\n            return block(x,memory,past,cross_past,position,cache)\n        block.attn._valid=valid\n        if hasattr(block,\'after_attention_adapter\'):\n            y,c=block.attn(block.attn_norm(x),past=past,position=position,use_cache=cache)\n            x=block.after_attention_adapter(x+block.drop(y));x=block.after_ff_adapter(x+block.drop(block.ff(block.ff_norm(x))))\n            return (x,c) if cache else x\n        return block(x,past,position,cache)\n    def encode(self,ids,valid):\n        x=self.backbone.embedding(ids)\n        for block in self.backbone.encoder:\n            def f(x,valid,block=block):return self.block(block,x,valid)\n            x=checkpoint(f,x,valid,use_reentrant=False) if self.training and self.checkpointing else f(x,valid)\n        return self.backbone.encoder_norm(x)\n    def hidden(self,ids,valid,source=None,source_valid=None,state=None,use_cache=False,source_width=None,source_lengths=None,compact=False):\n        state={} if state is None else state;pos=state.get(\'position\',0);x=self.backbone.embedding(ids)\n        if hasattr(self,\'prompt\') and pos==0:\n            x=torch.cat([self.prompt[None].expand(len(ids),-1,-1),x],1)\n            if valid is not None:valid=torch.cat([torch.ones(len(ids),self.prompt.shape[0],device=ids.device,dtype=torch.bool),valid],1)\n        if pos and \'valid\' in state:valid=torch.cat([state[\'valid\'],valid],1)\n        new={\'position\':pos+x.size(1),\'valid\':valid,\'self\':[]}\n        if hasattr(self.backbone,\'rnn\'):\n            # Fused LSTM stays outside checkpoint recomputation (cuDNN metadata safety).\n            if compact:\n                y,rnn=self.backbone.rnn(x)\n            elif pos==0:\n                # Skip masked source PADs in the recurrent state too, not only in attention.\n                width=source_width or self.cfg[\'source_tokens\'];lengths=source_lengths if source_lengths is not None else valid[:,:width].sum(1).cpu()\n                packed=nn.utils.rnn.pack_padded_sequence(x[:,:width],lengths,batch_first=True,enforce_sorted=False)\n                py,rnn=self.backbone.rnn(packed);sy,_=nn.utils.rnn.pad_packed_sequence(py,batch_first=True,total_length=width)\n                ty,rnn=self.backbone.rnn(x[:,width:],rnn);y=torch.cat([sy,ty],1)\n            else:y,rnn=self.backbone.rnn(x,state.get(\'rnn\'))\n            x=x+self.backbone.rnn_norm(y);new[\'rnn\']=rnn\n        memory=None\n        if self.native:\n            memory=state.get(\'memory\')\n            if memory is None:memory=self.encode(source,source_valid)\n            else:source_valid=state[\'source_valid\']\n            new.update(memory=memory,source_valid=source_valid,cross=[])\n        for i,block in enumerate(self.backbone.blocks):\n            if use_cache:\n                y=self.block(block,x,valid,memory,source_valid,pos,state.get(\'self\',[None]*len(self.backbone.blocks))[i],state.get(\'cross\',[None]*len(self.backbone.blocks))[i] if self.native else None,True)\n                x=y[0];new[\'self\'].append(y[1])\n                if self.native:new[\'cross\'].append(y[2])\n            else:\n                def f(x,valid,memory=memory,source_valid=source_valid,block=block):return self.block(block,x,valid,memory,source_valid)\n                x=checkpoint(f,x,valid,use_reentrant=False) if self.training and self.checkpointing else f(x,valid)\n        return self.backbone.final_norm(x),new\n    def report(self):return {\'total\':sum(p.numel() for p in self.parameters()),\'trainable\':sum(p.numel() for p in self.parameters() if p.requires_grad),\'architecture\':self.backbone.config.architecture,\'method\':self.cfg[\'method\']}\n\n'
(CODE / 's6_model.py').write_text(module_source, encoding="utf-8")
print("Wrote s6_model.py")


In [ ]:
# s6_run.py
module_source = '"""Distributed setup and synchronized FP16 updates, inherited from Phase 6."""\nimport json,os\nfrom pathlib import Path\nimport torch\nimport torch.distributed as dist\n\ndef atomic_save(p,x):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_name(p.name+\'.tmp\');torch.save(x,t);t.replace(p)\n\ndef initialize():\n    world=int(os.environ.get(\'WORLD_SIZE\',1));rank=int(os.environ.get(\'RANK\',0));local=int(os.environ.get(\'LOCAL_RANK\',0))\n    device=torch.device(f\'cuda:{local}\' if torch.cuda.is_available() else \'cpu\')\n    if device.type==\'cuda\':torch.cuda.set_device(device)\n    if world>1:\n        opts={\'device_id\':device} if device.type==\'cuda\' else {}\n        dist.init_process_group(\'nccl\' if device.type==\'cuda\' else \'gloo\',**opts)\n    return rank,world,device\n\ndef barrier(device):\n    if dist.is_initialized():dist.barrier(device_ids=[device.index] if device.type==\'cuda\' else None)\n\ndef all_finite(value,device):\n    flag=torch.isfinite(value.detach()).all().to(device=device,dtype=torch.int32)\n    if dist.is_initialized():dist.all_reduce(flag,op=dist.ReduceOp.MIN)\n    return bool(flag.item())\n\ndef finish_update(model,optimizer,scaler,params,device,stage):\n    """Unscale once at the effective-batch boundary; all ranks step or skip together."""\n    scaler.unscale_(optimizer)\n    gradients=[p.grad for p in params if p.grad is not None]\n    if not gradients:raise RuntimeError(\'No trainable gradients were produced.\')\n    norm=torch.nn.utils.get_total_norm(gradients,error_if_nonfinite=False)\n    if not bool(torch.isfinite(norm)) and all(bool(torch.isfinite(g).all()) for g in gradients):\n        # A float32 norm reduction itself can overflow even when every gradient element is finite.\n        norm=torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(g.detach().double()) for g in gradients]))\n    if not all_finite(norm,device):\n        bad=[n for n,p in model.named_parameters() if p.grad is not None and not bool(torch.isfinite(p.grad).all())][:8]\n        previous=float(scaler.get_scale());new_scale=previous*.5\n        print(json.dumps({\'event\':\'amp_gradient_overflow\',\'stage\':stage,\'rank\':dist.get_rank() if dist.is_initialized() else 0,\n                          \'scale_before\':previous,\'scale_after\':new_scale,\'nonfinite_gradient_parameters\':bad,\n                          \'optimizer_step\':\'skipped on all ranks; parameters unchanged\'}),flush=True)\n        if not scaler.is_enabled():raise FloatingPointError(f\'{stage}: nonfinite unscaled gradients with AMP disabled: {bad}\')\n        # Reset per-optimizer scaler state without updating weights, even if only a peer rank overflowed.\n        scaler.update(new_scale=new_scale);optimizer.zero_grad(set_to_none=True)\n        return float(\'inf\'),False\n    torch.nn.utils.clip_grads_with_norm_(params,1.,norm)\n    scaler.step(optimizer);scaler.update()\n    return float(norm),True\n'
(CODE / 's6_run.py').write_text(module_source, encoding="utf-8")
print("Wrote s6_run.py")


In [ ]:
# s6_workflow.py
module_source = '"""Official input discovery and submission validation from Phase 6."""\nimport os,shutil,zipfile\nfrom pathlib import Path\nimport torch\nfrom s6_data import FILES,atomic_json,read_json,file_hash,digest\n\ndef find_phase1(explicit=None):\n    if explicit:\n        p=Path(explicit)\n        if not (p/\'tokenizer.json\').is_file():raise FileNotFoundError(f\'No tokenizer.json under {p}\')\n        return p\n    preferred=Path(\'/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1\')\n    candidates=[preferred] if preferred.is_dir() else sorted({p.parent for p in Path(\'/kaggle/input\').rglob(\'tokenizer.json\') if p.parent.name==\'capstone_phase1\'})\n    if len(candidates)!=1:raise FileNotFoundError(\'Set PHASE1_ROOT to the unique original capstone_phase1 directory. Attach the Phase-1 outputs dataset.\')\n    return candidates[0]\n\ndef obtain_data(project,local_root=None,revision=None):\n    if not local_root:\n        candidates=sorted({p for p in Path(\'/kaggle/input\').rglob(\'Phase_6_Summ\') if (p/FILES[0]).is_file() and (p/FILES[1]).is_file()})\n        if len(candidates)==1:local_root=candidates[0]\n        elif len(candidates)>1:raise ValueError(\'Multiple Phase-6 datasets: set DATA_ROOT explicitly.\')\n    if local_root:\n        source=Path(local_root)\n        if (source/\'Phase_6_Summ\').is_dir():source/=\'Phase_6_Summ\'\n        if any(not (source/f).is_file() for f in FILES):raise FileNotFoundError(\'Need Phase_6_Summ/Train/train.json and Test/test.json.\')\n        atomic_json(Path(project)/\'data_source.json\',{\'kind\':\'attached_official_phase6\',\'root\':str(source),\'sha256\':{n:file_hash(source/n) for n in FILES}})\n        return str(source)\n    from huggingface_hub import HfApi,hf_hub_download\n    token=os.environ.get(\'HF_TOKEN\')\n    if not token:\n        try:\n            from kaggle_secrets import UserSecretsClient\n            token=UserSecretsClient().get_secret(\'HF_TOKEN\')\n        except Exception:token=None\n    root=Path(project)/\'data\';manifest=root/\'source.json\';old=read_json(manifest) if manifest.exists() else None\n    repo=\'Exploration-Lab/CS-781-Capstone\'\n    try:\n        commit=old[\'revision\'] if old else HfApi(token=token).dataset_info(repo,revision=revision).sha\n        if old and revision and revision!=commit:raise ValueError(\'Pinned revision differs. Choose a new PROJECT.\')\n        for name in FILES:\n            target=root/name\n            if target.exists() and old and old.get(\'sha256\',{}).get(name)==file_hash(target):continue\n            source=hf_hub_download(repo,\'Phase_6_Summ/\'+name,repo_type=\'dataset\',revision=commit,token=token)\n            target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)\n        atomic_json(manifest,{\'repo\':repo,\'revision\':commit,\'sha256\':{n:file_hash(root/n) for n in FILES}})\n    except Exception as e:raise RuntimeError(\'Attach official Phase_6_Summ data, or accept its Hugging Face terms, enable Kaggle Internet and add an HF_TOKEN secret.\') from e\n    return str(root)\n\ndef pack_prediction(project,name,pred_path,cache_path):\n    cache=torch.load(cache_path,map_location=\'cpu\',weights_only=False);expected={r[\'id\'] for r in cache[\'rows\'] if r[\'split\']==\'test\'}\n    pred=read_json(pred_path)\n    if not isinstance(pred,list) or len(pred)!=len(expected):raise ValueError(\'Wrong number of test predictions.\')\n    if len({r[\'id\'] for r in pred})!=len(pred) or {r[\'id\'] for r in pred}!=expected:raise ValueError(\'Submission IDs mismatch.\')\n    if any(set(r)!={\'id\',\'summary\'} or not isinstance(r[\'summary\'],str) or not r[\'summary\'].strip() for r in pred):raise ValueError(\'Invalid summary schema.\')\n    folder=Path(project)/\'submissions\'/name;atomic_json(folder/\'prediction.json\',pred);archive=folder.with_suffix(\'.zip\')\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:z.write(folder/\'prediction.json\',\'prediction.json\')\n    with zipfile.ZipFile(archive) as z:\n        if z.namelist()!=[\'prediction.json\']:raise ValueError(\'Incorrect ZIP root.\')\n    return {\'name\':name,\'submission_zip\':str(archive),\'prediction_sha256\':digest(sorted(pred,key=lambda r:r[\'id\'])),\'test_documents\':len(pred)}\n'
(CODE / 's6_workflow.py').write_text(module_source, encoding="utf-8")
print("Wrote s6_workflow.py")


In [ ]:
# s6_metrics.py
module_source = '"""Official-equivalent metrics; pretrained BERT is used for detached text evaluation only."""\nimport torch\n_scorer=None\ndef release():\n    global _scorer\n    import gc\n    _scorer=None;gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\ndef score(predictions,references,device=\'cpu\'):\n    global _scorer\n    from rouge_score import rouge_scorer,tokenizers\n    from bert_score import BERTScorer\n    from evaluate_phase6_official import lcs\n    if len(predictions)!=len(references) or not predictions:raise ValueError(\'Metric input lengths differ/empty.\')\n    if _scorer is None or str(_scorer.device)!=str(device):\n        _scorer=BERTScorer(lang=\'en\',batch_size=8,device=str(device),rescale_with_baseline=False,idf=False)\n    # The provided official evaluator uses bert_score.score(..., lang=\'en\'), without rescaling or IDF.\n    with torch.inference_mode():_,_,f=_scorer.score(predictions,references,batch_size=8)\n    rs=rouge_scorer.RougeScorer([\'rouge1\'],use_stemmer=True);tok=tokenizers.DefaultTokenizer(use_stemmer=True)\n    r1=[];rl=[]\n    for p,r in zip(predictions,references):\n        r1.append(rs.score(r,p)[\'rouge1\'].fmeasure);pp=tok.tokenize(p);rr=tok.tokenize(r)\n        rl.append(2*lcs(pp,rr)/max(1,len(pp)+len(rr)) if pp or rr else 1.)\n    return {\'bertscore_f1\':float(f.mean()),\'rouge1_f1\':sum(r1)/len(r1),\'rougeL_f1\':sum(rl)/len(rl),\n            \'documents\':len(predictions),\'bertscore_hash\':_scorer.hash,\'metric_model\':\'evaluation only; never part of generator/optimizer\'}\n'
(CODE / 's6_metrics.py').write_text(module_source, encoding="utf-8")
print("Wrote s6_metrics.py")


In [ ]:
# evaluate_phase6_official.py
module_source = '#!/usr/bin/env python3\n"""Local evaluator for Phase 6.\n\nRequires: rouge-score, bert-score, torch.\nTruth JSON: {"doc_id":"reference summary", ...}\nPrediction JSON: either {"doc_id":"summary", ...} or [{"id":"...","summary":"..."}, ...]\n"""\nimport argparse,json,sys\ntry:\n    from rouge_score import tokenizers, rouge_scorer\n    import bert_score\nexcept ImportError as e:\n    print(\'Missing dependency. Install requirements.txt first.\', file=sys.stderr); raise\n\ndef load(path):\n    with open(path,encoding=\'utf-8\') as f: return json.load(f)\n\ndef lcs(a,b):\n    if len(a)>len(b): a,b=b,a\n    masks={}\n    for i,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<i)\n    s=0\n    for t in b:\n        m=masks.get(t,0); x=m|s; y=(s<<1)|1; s=x & ~(x-y)\n    return s.bit_count() if hasattr(int,\'bit_count\') else bin(s).count(\'1\')\n\ndef read_pred(x):\n    if isinstance(x,dict): out=x\n    elif isinstance(x,list):\n        out={}\n        for r in x:\n            if not isinstance(r,dict) or \'id\' not in r or \'summary\' not in r: raise ValueError(\'Each prediction item must contain id and summary.\')\n            if r[\'id\'] in out: raise ValueError(f\'Duplicate prediction ID: {r["id"]}\')\n            out[r[\'id\']]=r[\'summary\']\n    else: raise ValueError(\'Prediction must be a JSON object or list of objects.\')\n    for k,v in out.items():\n        if not isinstance(k,str) or not isinstance(v,str) or not v.strip(): raise ValueError(f\'Invalid summary for ID {k!r}.\')\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 6 summaries on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); a=ap.parse_args()\n    t=load(a.truth); p=read_pred(load(a.prediction))\n    if not isinstance(t,dict) or not t: raise ValueError(\'Truth must be a non-empty JSON object.\')\n    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")\n    preds=[p[k].strip() for k in t]; refs=[t[k].strip() for k in t]\n    scorer=rouge_scorer.RougeScorer([\'rouge1\'],use_stemmer=True); tok=tokenizers.DefaultTokenizer(use_stemmer=True)\n    r1=rl=0.0\n    for pr,rf in zip(preds,refs):\n        s=scorer.score(rf,pr); r1+=s[\'rouge1\'].fmeasure\n        pt=tok.tokenize(pr); rt=tok.tokenize(rf)\n        if not pt and not rt: x=1.0\n        elif not pt or not rt: x=0.0\n        else:\n            z=lcs(pt,rt); pp=z/len(pt); rr=z/len(rt); x=0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)\n        rl+=x\n    n=len(t); r1/=n; rl/=n\n    try:\n        import torch; device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    except ImportError: device=None\n    _,_,f=bert_score.score(preds,refs,lang=\'en\',batch_size=8,verbose=False,**({\'device\':device} if device else {}))\n    print(f\'Documents: {n}\'); print(f\'BERTScore F1: {float(f.mean().item()):.6f}\'); print(f\'ROUGE-1: {r1:.6f}\'); print(f\'ROUGE-L: {rl:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n'
(CODE / 'evaluate_phase6_official.py').write_text(module_source, encoding="utf-8")
print("Wrote evaluate_phase6_official.py")


In [ ]:
# fast_data.py
module_source = '"""One shared, bounded preprocessing pass. No test target access."""\nimport collections,math\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom tokenizers import Tokenizer\n\nfrom s6_data import (FILES, Selector, atomic_json, digest, file_hash, normhash,\n                     sentences, special_ids, split_data, wordset)\nfrom s6_run import atomic_save\n\ndef choose_context(text,tokenizer,selector,budget,shortlist,sep):\n    ss=sentences(text)\n    ids=[e.ids for e in tokenizer.encode_batch(ss,add_special_tokens=False)]\n    if sum(map(len,ids))+len(sep)*max(0,len(ids)-1)<=budget:\n        return [t for i,row in enumerate(ids) for t in ((sep if i else [])+row)]\n    score=selector.model.predict(selector.features(ss))\n    selected={}\n    # Keep some opening facts and the actual tail of the closing fragment.\n    head=min(budget//10,len(ids[0]))\n    tail=min(budget//6,len(ids[-1]))\n    if len(ids)>1:\n        if head:selected[0]=ids[0][:head]\n        if tail:selected[len(ids)-1]=ids[-1][-tail:]\n    candidates=sorted((i for i,ts in enumerate(ids) if ts and i not in selected),\n                      key=lambda i:(-float(score[i]),i))[:shortlist]\n    bags={i:wordset(ss[i]) for i in set(candidates)|set(selected)}\n    def sim(i,j):\n        a,b=bags[i],bags[j]\n        return len(a&b)/max(1,len(a|b))\n    redundancy={i:max((sim(i,j) for j in selected),default=0.) for i in candidates}\n    remaining=budget-sum(len(x)+len(sep) for x in selected.values())\n    while candidates and remaining>len(sep):\n        i=max(candidates,key=lambda j:(float(score[j])-.15*redundancy[j],-j))\n        candidates.remove(i)\n        room=remaining-len(sep)\n        if len(ids[i])>room and remaining<budget//4 and len(ids[i])>2*room:continue\n        selected[i]=ids[i][:room]\n        remaining-=len(selected[i])+len(sep)\n        for j in candidates:redundancy[j]=max(redundancy[j],sim(i,j))\n    if not selected:return ids[0][:budget]\n    return [t for n,(_,row) in enumerate(sorted(selected.items())) for t in ((sep if n else [])+row)][:budget]\n\ndef encoded_rows(tok,texts):\n    for start in range(0,len(texts),256):\n        yield from (e.ids for e in tok.encode_batch(texts[start:start+256],add_special_tokens=False))\n\ndef prepare_fast(root,tokenizer_path,project,opts):\n    root=Path(root);tokenizer_path=Path(tokenizer_path);project=Path(project)\n    path=project/\'cache\'/\'phase6_fast.pt\'\n    keys=(\'seed\',\'source_tokens\',\'target_tokens\',\'dev_fraction\',\'dev_max\',\'train_pool\',\'selector_fit_docs\',\'selector_shortlist\',\'special_ids\')\n    identity=digest({\'data\':{f:file_hash(root/f) for f in FILES},\'tokenizer\':file_hash(tokenizer_path),\n                     \'settings\':{k:opts[k] for k in keys},\'preprocessor\':file_hash(__file__),\n                     \'parser\':file_hash(Path(__file__).with_name(\'s6_data.py\'))})\n    if path.exists():\n        value=torch.load(path,map_location=\'cpu\',weights_only=False)\n        if value[\'fingerprint\']!=identity:raise ValueError(\'Cache inputs changed. Set a new PROJECT.\')\n        if any(r[\'target\'] is not None or r[\'summary\'] is not None for r in value[\'rows\'] if r[\'split\']==\'test\'):\n            raise ValueError(\'Refusing test references in an existing cache.\')\n        if file_hash(tokenizer_path)!=value[\'tokenizer_sha256\']:raise ValueError(\'Tokenizer changed.\')\n        if value[\'tokenizer_path\']!=str(tokenizer_path):raise ValueError(\'Tokenizer mount changed. Use a new PROJECT.\')\n        atomic_json(project/\'cache\'/\'audit.json\',value[\'audit\'])\n        return path,value[\'audit\']\n    rows,audit=split_data(root,opts[\'seed\'],opts[\'dev_fraction\'],opts[\'dev_max\'])\n    train=sorted((r for r in rows if r[\'split\']==\'train\'),key=lambda r:digest([opts[\'seed\'],r[\'id\']]))\n    train=train[:opts[\'train_pool\']] if opts[\'train_pool\'] else train\n    ids={r[\'id\'] for r in train}\n    rows=[r for r in rows if r[\'split\']!=\'train\' or r[\'id\'] in ids]\n    print(f\'Preprocessing {len(train):,} training, {sum(r["split"]=="dev" for r in rows):,} dev and {sum(r["split"]=="test" for r in rows):,} test documents.\',flush=True)\n    tok=Tokenizer.from_file(str(tokenizer_path));tok.no_padding();tok.no_truncation()\n    special=special_ids(tok,opts[\'special_ids\'])\n    blocked=sorted(set([special[\'pad\'],special[\'bos\']]+[r[\'id\'] for r in __import__(\'json\').loads(tok.to_str()).get(\'added_tokens\',[]) if r.get(\'special\') and r[\'id\']!=special[\'eos\']]))\n    fit_rows=train[:opts[\'selector_fit_docs\']]\n    selector=Selector(opts[\'seed\']).fit(fit_rows,passes=1)\n    refs=[r for r in rows if r[\'split\']!=\'test\']\n    labels={r[\'id\']:ts for r,ts in zip(refs,encoded_rows(tok,[r[\'summary\'] for r in refs]))}\n    lengths=np.asarray([len(labels[r[\'id\']]) for r in train])\n    cap=min(opts[\'target_tokens\'],max(64,int(np.quantile(lengths,.95))+1))\n    prefix=tok.encode(\'Summarize the legal judgment, including the material facts, legal issue, reasoning and final decision.\\nJudgment:\\n\',add_special_tokens=False).ids\n    suffix=tok.encode(\'\\nSummary:\\n\',add_special_tokens=False).ids\n    sep=tok.encode(\'\\n\',add_special_tokens=False).ids\n    room=opts[\'source_tokens\']-len(prefix)-len(suffix)\n    if room<32:raise ValueError(\'SOURCE_TOKENS is too small for the original tokenizer and prompt.\')\n    data=[]\n    for i,r in enumerate(rows,1):\n        sid=r[\'id\'];target=None if r[\'split\']==\'test\' else labels[sid][:cap-1]+[special[\'eos\']]\n        source=prefix+choose_context(r[\'document\'],tok,selector,room,opts[\'selector_shortlist\'],sep)+suffix\n        data.append({\'id\':sid,\'split\':r[\'split\'],\'source\':source,\'target\':target,\'summary\':r[\'summary\'],\'document_sha256\':normhash(r[\'document\'])})\n        if i%500==0:print(f\'Cached {i:,}/{len(rows):,} documents\',flush=True)\n    audit.update(counts=dict(collections.Counter(r[\'split\'] for r in data)),\n                 selector_fit_documents=len(fit_rows),selector_shortlist=opts[\'selector_shortlist\'],\n                 source_limit=opts[\'source_tokens\'],target_limit=cap,\n                 training_summary_tokens_p50_p90_p95=[round(float(np.quantile(lengths,q)),1) for q in (.5,.9,.95)],\n                 train_references_clipped=int((lengths>=cap).sum()),\n                 train_references_clipped_fraction=round(float((lengths>=cap).mean()),4))\n    value={\'rows\':data,\'fingerprint\':identity,\'tokenizer_path\':str(tokenizer_path),\'tokenizer_sha256\':file_hash(tokenizer_path),\n           \'special\':special,\'generation_blocked_ids\':blocked,\'source_tokens\':opts[\'source_tokens\'],\'target_tokens\':cap,\'audit\':audit}\n    atomic_save(path,value);atomic_json(project/\'cache\'/\'audit.json\',audit)\n    return path,audit\n'
(CODE / 'fast_data.py').write_text(module_source, encoding="utf-8")
print("Wrote fast_data.py")


In [ ]:
# fast_model.py
module_source = '"""Dynamically batched conditional LM with chunked CE/KD and greedy KV decoding."""\nimport math\nfrom collections import defaultdict\n\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\nfrom tokenizers import Tokenizer\n\nfrom s6_model import Summarizer, load_parent, merged_adapters\n\ndef pad(rows,fill,width=None):\n    if width is None:width=max(map(len,rows))\n    return torch.tensor([row+[fill]*(width-len(row)) for row in rows],dtype=torch.long)\n\ndef align(n,bound,multiple=32):\n    return min(bound,math.ceil(n/multiple)*multiple)\n\ndef collation(rows,cache,device,train=True,native=False):\n    pad_id=cache[\'special\'][\'pad\'];bos=cache[\'special\'][\'bos\']\n    ss=[r[\'source\'] for r in rows];lengths=torch.tensor(list(map(len,ss)),dtype=torch.long)\n    if train:\n        ys=[r[\'target\'] for r in rows]\n        if any(not y for y in ys):raise ValueError(\'Missing target in supervised batch.\')\n        target=torch.tensor([tok for y in ys for tok in y],dtype=torch.long).to(device)\n        row_index=torch.tensor([i for i,y in enumerate(ys) for _ in y],dtype=torch.long).to(device)\n        inputs=[[bos]+y[:-1] for y in ys]\n        if not native:\n            seq=[s+x for s,x in zip(ss,inputs)]\n            width=align(max(map(len,seq)),cache[\'source_tokens\']+cache[\'target_tokens\'])\n            positions=torch.tensor([j for src,y in zip(ss,ys) for j in range(len(src),len(src)+len(y))],\n                                   dtype=torch.long).to(device)\n            return {\'ids\':pad(seq,pad_id,width).to(device),\'target\':target,\'rows\':row_index,\n                    \'positions\':positions,\'token_count\':target.numel()}\n        ti=pad(inputs,pad_id).to(device)\n        positions=torch.tensor([j for y in ys for j in range(len(y))],dtype=torch.long).to(device)\n    width=align(max(map(len,ss)),cache[\'source_tokens\'])\n    source=pad(ss,pad_id,width).to(device)\n    valid=(torch.arange(width)[None]<lengths[:,None]).to(device)\n    batch={\'source\':source,\'source_valid\':valid,\'lengths\':lengths}\n    if train:batch.update(ids=ti,target=target,rows=row_index,positions=positions,token_count=target.numel())\n    return batch\n\nclass FastSummarizer(Summarizer):\n    def target_states(self,batch):\n        if self.native:\n            h,_=self.hidden(batch[\'ids\'],None,batch[\'source\'],batch[\'source_valid\'],compact=True)\n        else:\n            h,_=self.hidden(batch[\'ids\'],None,compact=True)\n        offset=self.prompt.shape[0] if hasattr(self,\'prompt\') else 0\n        return h[batch[\'rows\'],batch[\'positions\']+offset]\n    def forward(self,batch,teacher_states=None,teacher_weight=None):\n        h=self.target_states(batch);y=batch[\'target\']\n        step=self.cfg[\'projection_chunk\'];loss=h.new_zeros((),dtype=torch.float32)\n        temp=self.cfg[\'kd_temperature\'];weight=self.cfg[\'kd_weight\']\n        def ce(hidden,gold,other=None):\n            logits=self.backbone.project(hidden).float()\n            with torch.autocast(device_type=hidden.device.type,enabled=False):\n                nll=F.cross_entropy(logits,gold,reduction=\'sum\',label_smoothing=self.cfg[\'label_smoothing\'])\n            if other is None:return nll\n            with torch.no_grad():tl=F.linear(other,teacher_weight).float()\n            with torch.autocast(device_type=hidden.device.type,enabled=False):\n                kl=F.kl_div(F.log_softmax(logits/temp,dim=-1),F.softmax(tl/temp,dim=-1),reduction=\'sum\')\n            return (1.-weight)*nll+weight*temp**2*kl\n        for start in range(0,len(y),step):\n            stop=start+step\n            args=(h[start:stop],y[start:stop],None if teacher_states is None else teacher_states[start:stop])\n            loss=loss+(checkpoint(ce,*args,use_reentrant=False) if self.training else ce(*args))\n        return loss/batch[\'token_count\']\n\ndef build(cfg,cache,device):\n    parent=load_parent(cfg[\'parent\'],cache[\'tokenizer_sha256\'],cfg.get(\'allow_tiny\',False))\n    if parent.config.architecture!=cfg[\'architecture\']:raise ValueError(\'Parent architecture mismatch.\')\n    s,t=cache[\'source_tokens\'],cache[\'target_tokens\']\n    needed=max(s,t+1) if cfg[\'architecture\']==\'encoder_decoder\' else s+t+cfg.get(\'prompt_tokens\',0)+1\n    if needed>parent.config.max_positions:raise ValueError(f\'Need {needed} positions; Phase 1 parent supports {parent.config.max_positions}.\')\n    return FastSummarizer(parent,cfg).to(device)\n\n@torch.inference_mode()\ndef greedy(model,rows,cache,device,decode,first_blocked=()):\n    batch=collation(rows,cache,device,train=False);src=batch[\'source\'];sv=batch[\'source_valid\']\n    bos=torch.full((len(rows),1),cache[\'special\'][\'bos\'],device=device,dtype=torch.long)\n    ids=bos if model.native else torch.cat([src,bos],1)\n    valid=torch.ones_like(ids,dtype=torch.bool) if model.native else torch.cat([sv,torch.ones_like(bos,dtype=torch.bool)],1)\n    blocked=cache[\'generation_blocked_ids\'];eos=cache[\'special\'][\'eos\']\n    max_new=decode[\'max_new_tokens\'];min_new=decode[\'min_new_tokens\'];n=decode[\'no_repeat_ngram\']\n    sequences=[[] for _ in rows];grams=[defaultdict(set) for _ in rows];done=[False]*len(rows);state=None\n    for step in range(max_new):\n        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n            h,state=model.hidden(ids,valid,src if model.native else None,sv if model.native else None,state,True,\n                                 source_width=src.shape[1],source_lengths=batch[\'lengths\'])\n            logits=model.backbone.project(h[:,-1])\n        if blocked:logits[:,blocked]=float(\'-inf\')\n        if step==0 and first_blocked:logits[:,list(first_blocked)]=float(\'-inf\')\n        if step<min_new:logits[:,eos]=float(\'-inf\')\n        if n>=2:\n            ban=[(i,t) for i,seq in enumerate(sequences) if not done[i] and len(seq)>=n-1\n                 for t in grams[i].get(tuple(seq[-(n-1):]),())]\n            if ban:\n                index,values=zip(*ban);logits[list(index),list(values)]=float(\'-inf\')\n        ids=logits.argmax(-1,keepdim=True)\n        for i,token in enumerate(ids[:,0].tolist()):\n            if done[i]:continue\n            if token==eos:done[i]=True;continue\n            seq=sequences[i]\n            if n>=2 and len(seq)>=n-1:grams[i][tuple(seq[-(n-1):])].add(token)\n            seq.append(token)\n        if all(done):break\n        valid=torch.ones_like(ids,dtype=torch.bool)\n    return sequences\n\n@torch.inference_mode()\ndef predict(model,rows,cache,device,decode,batch_size):\n    tok=Tokenizer.from_file(cache[\'tokenizer_path\']);result=[];first_blocked=None\n    rows=sorted(rows,key=lambda r:(len(r[\'source\']),r[\'id\']))\n    model.eval()\n    with merged_adapters(model):\n        for start in range(0,len(rows),batch_size):\n            local=rows[start:start+batch_size]\n            while True:\n                # A smaller generation batch does not change the IDs covered.\n                try:\n                    seqs=greedy(model,local,cache,device,decode)\n                    break\n                except torch.OutOfMemoryError:\n                    if len(local)==1:raise\n                    seqs=None\n                    break\n            if seqs is None:\n                if device.type==\'cuda\':torch.cuda.empty_cache()\n                seqs=[greedy(model,[r],cache,device,decode)[0] for r in local]\n            for row,seq in zip(local,seqs):\n                text=tok.decode(seq,skip_special_tokens=True).strip()\n                if not text:\n                    if first_blocked is None:\n                        first_blocked=[i for i in range(tok.get_vocab_size()) if not tok.decode([i],skip_special_tokens=True).strip()]\n                    retry=greedy(model,[row],cache,device,decode,first_blocked)\n                    text=tok.decode(retry[0],skip_special_tokens=True).strip()\n                if not text:raise ValueError(f\'Empty generated summary for {row["id"]!r}. Check the Phase 1 tokenizer and model.\')\n                result.append({\'id\':row[\'id\'],\'summary\':text})\n            if (start//batch_size+1)%25==0:print(f\'Generated {len(result):,}/{len(rows):,} summaries.\',flush=True)\n    return result\n'
(CODE / 'fast_model.py').write_text(module_source, encoding="utf-8")
print("Wrote fast_model.py")


In [ ]:
# fast_run.py
module_source = '"""DDP worker for bounded training and validation by teacher-forced dev loss."""\nimport argparse,contextlib,json,math,random,time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\n\nfrom fast_model import build,collation,predict\nfrom s6_data import atomic_json,digest,file_hash,read_json\nfrom s6_run import all_finite,atomic_save,barrier,finish_update,initialize\n\ndef epoch_batches(rows,cfg,epoch,rank,world):\n    """Shuffle globally, rotate through the pool, then bucket short ranges by length."""\n    base=random.Random(cfg[\'seed\']).sample(range(len(rows)),len(rows))\n    stride=cfg[\'micro_batch\']*world\n    per_epoch=cfg[\'steps_per_epoch\']*cfg[\'accumulation\']*stride\n    start=(epoch*per_epoch)%len(base)\n    order=(base[start:]+base[:start])[:min(len(base),per_epoch)]\n    random.Random(cfg[\'seed\']+epoch+1).shuffle(order)\n    bucket=16*stride\n    local=[]\n    for j in range(0,len(order),bucket):\n        nearby=sorted(order[j:j+bucket],key=lambda i:len(rows[i][\'source\'])+len(rows[i][\'target\']))\n        for z in range(0,len(nearby),stride):\n            chosen=nearby[z:z+stride]\n            n=len(chosen)\n            chosen += [chosen[k%n] for k in range(stride-n)]\n            off=rank*cfg[\'micro_batch\']\n            local.append([rows[i] for i in chosen[off:off+cfg[\'micro_batch\']]])\n    random.Random(cfg[\'seed\']+epoch+191).shuffle(local)\n    return local\n\ndef autocast(device):\n    return torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type==\'cuda\')\n\n@torch.inference_mode()\ndef dev_loss(model,rows,cache,cfg,device,rank,world):\n    model.eval()\n    result=torch.zeros(2,device=device,dtype=torch.float64)\n    rows=sorted(rows[rank::world],key=lambda r:len(r[\'source\'])+len(r[\'target\']))\n    for i in range(0,len(rows),cfg[\'micro_batch\']):\n        batch=collation(rows[i:i+cfg[\'micro_batch\']],cache,device,native=model.native)\n        with autocast(device):value=model(batch)\n        result[0]+=value.detach().double()*batch[\'token_count\']\n        result[1]+=batch[\'token_count\']\n    if world>1:dist.all_reduce(result)\n    value=float(result[0]/result[1].clamp_min(1))\n    if not math.isfinite(value):raise FloatingPointError(\'Nonfinite teacher-forced development loss.\')\n    return value\n\ndef lineage(cfg,cache,world):\n    result={\'configuration\':cfg,\'cache_fingerprint\':cache[\'fingerprint\'],\n            \'tokenizer_sha256\':cache[\'tokenizer_sha256\'],\'parent_sha256\':file_hash(cfg[\'parent\']),\n            \'world_size\':world,\'code_sha256\':{p.name:file_hash(p) for p in Path(__file__).parent.glob(\'*.py\')}}\n    if cfg.get(\'teacher_config\'):\n        tc=read_json(cfg[\'teacher_config\'])\n        result[\'teacher_config_sha256\']=file_hash(cfg[\'teacher_config\'])\n        result[\'teacher_weights_sha256\']=file_hash(Path(tc[\'run\'])/\'best.pt\')\n    if cfg.get(\'warm_checkpoint\'):\n        result[\'warm_weights_sha256\']=file_hash(cfg[\'warm_checkpoint\'])\n        result[\'warm_cache_sha256\']=file_hash(cfg[\'warm_cache\'])\n    return result\n\ndef warm_start(model,cfg,cache,own):\n    path=cfg.get(\'warm_checkpoint\')\n    if not path:return\n    old=torch.load(path,map_location=\'cpu\',weights_only=False)\n    oldmeta=old.get(\'lineage\',{})\n    oldcfg=oldmeta.get(\'config\',{})\n    if cfg[\'name\']!=\'04_hybrid_lora\' or any(oldcfg.get(k)!=cfg[k] for k in (\'architecture\',\'method\',\'rank\',\'alpha\')):\n        raise ValueError(\'Warm start requires the original Phase 6 04_hybrid_lora checkpoint (rank 16, alpha 32).\')\n    if oldmeta.get(\'parent_sha256\')!=own[\'parent_sha256\'] or oldmeta.get(\'tokenizer_sha256\')!=cache[\'tokenizer_sha256\']:\n        raise ValueError(\'Warm checkpoint Phase 1 parent or tokenizer does not match.\')\n    oldcache=torch.load(cfg[\'warm_cache\'],map_location=\'cpu\',weights_only=False)\n    if oldcache.get(\'fingerprint\')!=oldmeta.get(\'data_fingerprint\'):\n        raise ValueError(\'Warm checkpoint and original v4 token cache do not match.\')\n    old_train={r[\'id\'] for r in oldcache[\'rows\'] if r[\'split\']==\'train\'}\n    old_train_text={r.get(\'document_sha256\') for r in oldcache[\'rows\'] if r[\'split\']==\'train\'}\n    if any(r[\'id\'] in old_train or r[\'document_sha256\'] in old_train_text for r in cache[\'rows\'] if r[\'split\']!=\'train\'):\n        raise ValueError(\'Warm checkpoint previously trained on a current dev or test document.\')\n    model.load_state_dict(old[\'model\'],strict=True)\n\ndef rng_snapshot(device):\n    return {\'py\':random.getstate(),\'np\':np.random.get_state(),\'torch\':torch.get_rng_state(),\n            \'cuda\':torch.cuda.get_rng_state(device) if device.type==\'cuda\' else None}\ndef restore_rng(x,device):\n    random.setstate(x[\'py\']);np.random.set_state(x[\'np\']);torch.set_rng_state(x[\'torch\'])\n    if device.type==\'cuda\':torch.cuda.set_rng_state(x[\'cuda\'],device)\n\ndef train(model,cfg,cache,rank,world,device,sig,own):\n    run=Path(cfg[\'run\']);last=run/\'last.pt\';done=run/\'train_complete.json\'\n    if done.exists():\n        old=read_json(done)\n        if old[\'signature\']!=sig or old[\'checkpoint_sha256\']!=file_hash(run/\'best.pt\'):raise ValueError(\'Completed training input changed. Use a new PROJECT.\')\n        if rank==0:print(\'Reusing verified trained checkpoint:\',cfg[\'name\'],flush=True)\n        return\n    train_rows=[r for r in cache[\'rows\'] if r[\'split\']==\'train\']\n    dev=sorted([r for r in cache[\'rows\'] if r[\'split\']==\'dev\'],key=lambda r:digest([cfg[\'seed\'],r[\'id\']]))[:cfg[\'dev_loss_cases\']]\n    if not train_rows or not dev:raise ValueError(\'Missing training or held-out records.\')\n    teacher=None\n    if cfg.get(\'teacher_config\'):\n        tc=read_json(cfg[\'teacher_config\'])\n        weights=torch.load(Path(tc[\'run\'])/\'best.pt\',map_location=\'cpu\',weights_only=False)\n        if weights[\'signature\']!=digest(lineage(tc,cache,world)):raise ValueError(\'Teacher configuration or lineage mismatch.\')\n        teacher=build(tc,cache,device);teacher.load_state_dict(weights[\'model\'],strict=True);teacher.requires_grad_(False);teacher.eval()\n        del weights\n    if cfg.get(\'warm_checkpoint\') and not last.exists():\n        warm_start(model,cfg,cache,own)\n        if rank==0:print(\'Loaded previous Model 04 weights; using a fresh optimizer and budget.\',flush=True)\n    params=[p for p in model.parameters() if p.requires_grad]\n    wrapped=DDP(model,device_ids=[device.index] if device.type==\'cuda\' else None,broadcast_buffers=False,\n                gradient_as_bucket_view=True) if world>1 else model\n    optimizer=torch.optim.AdamW(params,lr=cfg[\'lr\'],weight_decay=0.01,\n                                **({\'fused\':True} if device.type==\'cuda\' else {}))\n    scaler=torch.amp.GradScaler(device.type,init_scale=1024.,enabled=device.type==\'cuda\')\n    updates=0;best=float(\'inf\');best_epoch=0;begin=0;stale=0;seconds=0.;stop=False;skips=0\n    if last.exists():\n        saved=torch.load(last,map_location=\'cpu\',weights_only=False)\n        if saved[\'signature\']!=sig:raise ValueError(\'Resume inputs differ. Use a new PROJECT.\')\n        model.load_state_dict(saved[\'model\'],strict=True);optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\'])\n        best=saved[\'best\'];best_epoch=saved[\'best_epoch\'];begin=saved[\'epoch\'];updates=saved[\'updates\'];stale=saved[\'stale\'];seconds=saved[\'train_seconds\'];stop=saved[\'stop\']\n        restore_rng(saved[\'rng\'][rank],device);del saved\n    num_micro=len(epoch_batches(train_rows,cfg,0,rank,world))\n    per_epoch=math.ceil(num_micro/cfg[\'accumulation\']);total=per_epoch*cfg[\'epochs\']\n    if rank==0:print(json.dumps({\'run\':cfg[\'name\'],\'max_optimizer_updates\':total,\'training_pool\':len(train_rows),\n                                 \'max_training_minutes\':cfg[\'max_train_minutes\'],\'parameters\':model.report()}),flush=True)\n    for epoch in range(begin,cfg[\'epochs\']):\n        if stop:break\n        model.train()\n        batches=epoch_batches(train_rows,cfg,epoch,rank,world)\n        start=time.perf_counter();running=torch.zeros((),device=device);attempts=0\n        optimizer.zero_grad(set_to_none=True)\n        for i,rows in enumerate(batches):\n            batch=collation(rows,cache,device,native=model.native)\n            sync=(i+1)%cfg[\'accumulation\']==0 or i+1==len(batches)\n            divisor=min(cfg[\'accumulation\'],len(batches)-(i//cfg[\'accumulation\'])*cfg[\'accumulation\'])\n            context=wrapped.no_sync() if world>1 and not sync else contextlib.nullcontext()\n            with context:\n                with autocast(device):\n                    # Teacher uses the same gold-prefix positions, never generates long rollouts.\n                    do_kd=teacher is not None and (epoch*len(batches)+i)%cfg[\'kd_every\']==0\n                    with torch.no_grad():\n                        th=teacher.target_states(batch) if do_kd else None\n                    loss=wrapped(batch,th,teacher.backbone.embedding.weight if do_kd else None)\n                    scaled=loss/divisor\n                scaler.scale(scaled).backward()\n            running+=loss.detach()\n            if not sync:continue\n            attempts+=1\n            warm=max(1,math.ceil(total*.06))\n            progress=max(0,updates-warm)/max(1,total-warm)\n            factor=(updates+1)/warm if updates<warm else .15+.85*.5*(1+math.cos(math.pi*progress))\n            for group in optimizer.param_groups:group[\'lr\']=cfg[\'lr\']*factor\n            _,stepped=finish_update(model,optimizer,scaler,params,device,f\'epoch{epoch+1}-batch{i+1}\')\n            optimizer.zero_grad(set_to_none=True)\n            updates+=int(stepped);skips=0 if stepped else skips+1\n            if skips>=12:raise FloatingPointError(\'Repeated FP16 overflow. Inspect the logged parameters.\')\n            spent=seconds+time.perf_counter()-start\n            if rank==0 and (attempts in (1,10) or attempts%25==0):\n                eta=(time.perf_counter()-start)/attempts*max(0,total-updates)/60\n                if cfg[\'max_train_minutes\']:eta=min(eta,max(0,cfg[\'max_train_minutes\']-spent/60))\n                print(json.dumps({\'epoch\':epoch+1,\'updates\':updates,\'running_training_loss\':float(running/(i+1)),\n                                  \'train_minutes_elapsed\':round(spent/60,2),\'remaining_train_minutes_estimate\':round(eta,2)}),flush=True)\n            timed=torch.tensor([int(rank==0 and bool(updates) and bool(cfg[\'max_train_minutes\']) and spent>=60*cfg[\'max_train_minutes\'])],device=device)\n            if world>1 and cfg[\'max_train_minutes\']:dist.broadcast(timed,src=0)\n            stop=bool(timed.item())\n            if stop:break\n        seconds+=time.perf_counter()-start\n        if updates==0:raise FloatingPointError(\'No finite optimizer update completed; the model was not trained.\')\n        value=dev_loss(model,dev,cache,cfg,device,rank,world)\n        improved=value<best-1e-5\n        if improved:best=value;best_epoch=epoch+1;stale=0\n        else:stale+=1\n        stop=stop or stale>=cfg[\'patience\']\n        rng=rng_snapshot(device);rngs=[None]*world\n        if world>1:dist.all_gather_object(rngs,rng)\n        else:rngs=[rng]\n        if rank==0:\n            saved={\'model\':model.state_dict(),\'signature\':sig,\'lineage\':own,\'epoch\':epoch+1,\'best_epoch\':best_epoch,\n                   \'updates\':updates,\'best\':best,\'train_seconds\':seconds,\'stale\':stale,\'stop\':stop,\'rng\':rngs}\n            if improved:atomic_save(run/\'best.pt\',saved)\n            atomic_save(last,{**saved,\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict()})\n            print(json.dumps({\'epoch\':epoch+1,\'dev_token_loss\':value,\'best_epoch\':best_epoch,\'minutes_training_only\':round(seconds/60,2),\n                              \'training_budget_reached\':bool(timed.item())}),flush=True)\n        barrier(device)\n    if rank==0:\n        atomic_json(done,{\'signature\':sig,\'best_epoch\':best_epoch,\'best_dev_loss\':best,\'training_seconds\':seconds,\n                          \'optimizer_updates\':updates,\'checkpoint_sha256\':file_hash(run/\'best.pt\'),\'parameters\':model.report()})\n    barrier(device)\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--config\',required=True);ap.add_argument(\'--action\',choices=(\'train\',\'generate\'),required=True)\n    ap.add_argument(\'--split\',choices=(\'dev\',\'test\'),default=\'test\');ap.add_argument(\'--output\');ap.add_argument(\'--limit\',type=int)\n    args=ap.parse_args()\n    torch.set_num_threads(2)\n    cfg=read_json(args.config);cache=torch.load(cfg[\'cache\'],map_location=\'cpu\',weights_only=False)\n    rank,world,device=initialize();random.seed(cfg[\'seed\']+rank);np.random.seed(cfg[\'seed\']+rank);torch.manual_seed(cfg[\'seed\'])\n    if device.type==\'cuda\':torch.backends.cudnn.benchmark=False\n    run=Path(cfg[\'run\']);run.mkdir(parents=True,exist_ok=True)\n    own=lineage(cfg,cache,world);sig=digest(own);model=build(cfg,cache,device)\n    if args.action==\'train\':\n        train(model,cfg,cache,rank,world,device,sig,own)\n    else:\n        if not args.output:raise ValueError(\'--output is required for generation.\')\n        if args.split==\'test\' and args.limit is not None:raise ValueError(\'Test generation must cover every official ID.\')\n        done=read_json(run/\'train_complete.json\')\n        if done[\'signature\']!=sig or done[\'checkpoint_sha256\']!=file_hash(run/\'best.pt\'):raise ValueError(\'Checkpoint provenance mismatch.\')\n        ck=torch.load(run/\'best.pt\',map_location=\'cpu\',weights_only=False)\n        model.load_state_dict(ck[\'model\'],strict=True);del ck\n        rows=sorted([r for r in cache[\'rows\'] if r[\'split\']==args.split],key=lambda r:digest([cfg[\'seed\'],r[\'id\']]))\n        if args.limit is not None:rows=rows[:args.limit]\n        local=predict(model,rows[rank::world],cache,device,cfg[\'decode\'],cfg[\'generation_batch\'])\n        atomic_json(args.output+f\'.rank{rank}.json\',local);barrier(device)\n        if rank==0:\n            combined=[item for i in range(world) for item in read_json(args.output+f\'.rank{i}.json\')]\n            index={r[\'id\']:r for r in combined}\n            if len(rows)!=len(combined) or set(index)!={r[\'id\'] for r in rows}:raise ValueError(\'Incomplete or duplicate generation IDs.\')\n            atomic_json(args.output,[index[r[\'id\']] for r in rows])\n            for i in range(world):Path(args.output+f\'.rank{i}.json\').unlink()\n        barrier(device)\n    if dist.is_initialized():dist.destroy_process_group()\n\nif __name__==\'__main__\':main()\n'
(CODE / 'fast_run.py').write_text(module_source, encoding="utf-8")
print("Wrote fast_run.py")


In [ ]:
# fast_workflow.py
module_source = '"""Notebook orchestration: four experiments, independent submission ZIPs, optional dev metrics."""\nimport os,socket,subprocess,sys,time,zipfile\nfrom pathlib import Path\n\nimport torch\nfrom fast_run import lineage\nfrom s6_data import atomic_json,digest,file_hash,read_json\nfrom s6_workflow import pack_prediction\n\nSPECS=[\n    (\'04_hybrid_lora\',\'hybrid\',\'lora\',2e-4),\n    (\'05_decoder_prompt\',\'decoder\',\'prompt\',3e-3),\n    (\'06_decoder_student_kd\',\'decoder\',\'kd\',8e-5),\n    (\'07_encoder_decoder_cross_attention\',\'encoder_decoder\',\'cross_only\',1.5e-4),\n]\ndef configurations(project,phase1,cache_path,opts,chosen):\n    project=Path(project);phase1=Path(phase1)\n    cache=torch.load(cache_path,map_location=\'cpu\',weights_only=False)\n    known={x[0] for x in SPECS}\n    if not chosen or len(set(chosen))!=len(chosen) or not set(chosen)<=known:raise ValueError(f\'Select unique names from {sorted(known)}.\')\n    needed=set(chosen)\n    if SPECS[2][0] in needed:needed.add(SPECS[0][0])\n    configs={}\n    for name,arch,method,lr in SPECS:\n        if name not in needed:continue\n        parent=phase1/\'runs\'/f\'main_{arch}\'/\'best_model.pt\'\n        if not parent.is_file():raise FileNotFoundError(f\'Missing original Phase 1 parent: {parent}\')\n        teacher=str(project/\'configs\'/f\'{SPECS[0][0]}.json\') if method==\'kd\' else None\n        cfg={k:opts[k] for k in (\'seed\',\'epochs\',\'steps_per_epoch\',\'micro_batch\',\'accumulation\',\'generation_batch\',\'max_train_minutes\',\n                                 \'dev_loss_cases\',\'kd_temperature\',\'kd_weight\',\'kd_every\',\'checkpointing\')}\n        target=cache[\'target_tokens\']\n        cfg.update(name=name,architecture=arch,method=method,lr=lr,parent=str(parent),cache=str(cache_path),\n                   run=str(project/\'runs\'/name),rank=16,alpha=32,prompt_tokens=32 if method==\'prompt\' else 0,\n                   source_tokens=cache[\'source_tokens\'],dropout=.05,projection_chunk=128,label_smoothing=.05,patience=1,\n                   decode={\'min_new_tokens\':min(48,max(1,target//5)),\'max_new_tokens\':target,\'no_repeat_ngram\':4},\n                   teacher_config=teacher,\n                   warm_checkpoint=opts[\'warm_checkpoint\'] if method==\'lora\' else None,\n                   warm_cache=opts[\'warm_cache\'] if method==\'lora\' else None)\n        if opts.get(\'allow_tiny\'):cfg[\'allow_tiny\']=True\n        path=project/\'configs\'/f\'{name}.json\'\n        if path.exists() and read_json(path)!=cfg:raise ValueError(f\'{name}: config changed. Use a new PROJECT or restore the old settings.\')\n        atomic_json(path,cfg);configs[name]=path\n    return configs\n\ndef launch(path,action,world,**kwargs):\n    cfg=read_json(path)\n    worker=Path(__file__).with_name(\'fast_run.py\')\n    with socket.socket() as sock:\n        sock.bind((\'127.0.0.1\',0));port=sock.getsockname()[1]\n    cmd=[sys.executable,\'-m\',\'torch.distributed.run\',\'--nnodes=1\',\'--master_addr=127.0.0.1\',\n         f\'--master_port={port}\',f\'--nproc_per_node={world}\',str(worker),\n         \'--config\',str(path),\'--action\',action]\n    for k,v in kwargs.items():cmd.extend([\'--\'+k.replace(\'_\',\'-\'),str(v)])\n    env={**os.environ,\'TOKENIZERS_PARALLELISM\':\'false\',\'OMP_NUM_THREADS\':\'2\',\'PYTHONUNBUFFERED\':\'1\'}\n    log=Path(cfg[\'run\'])/f"{action}_{kwargs.get(\'split\',\'\')}.log";log.parent.mkdir(parents=True,exist_ok=True)\n    with log.open(\'a\',encoding=\'utf-8\') as f,subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env,bufsize=1) as p:\n        for line in p.stdout:\n            print(line,end=\'\',flush=True);f.write(line);f.flush()\n        if p.wait():raise RuntimeError(f\'{action} failed for {cfg["name"]}. Log: {log}. Earlier ZIPs remain saved.\')\n\ndef run_candidates(project,configs,chosen,world):\n    project=Path(project);results=[];started=time.perf_counter()\n    for name,path in configs.items():\n        cfg=read_json(path);run=Path(cfg[\'run\'])\n        done=run/\'train_complete.json\'\n        cache=torch.load(cfg[\'cache\'],map_location=\'cpu\',weights_only=False)\n        signature=digest(lineage(cfg,cache,world));del cache\n        if not done.exists():\n            launch(path,\'train\',world)\n        else:\n            state=read_json(done)\n            if state[\'signature\']!=signature or state[\'checkpoint_sha256\']!=file_hash(run/\'best.pt\'):\n                raise ValueError(f\'{name}: inputs or weights changed. Use a new PROJECT.\')\n        if name not in chosen:continue\n        receipt=run/\'submission_complete.json\'\n        if receipt.exists():\n            old=read_json(receipt);result=old[\'result\']\n            if old[\'signature\']!=signature or file_hash(result[\'submission_zip\'])!=old[\'zip_sha256\']:\n                raise ValueError(f\'{name}: previous ZIP or signature changed.\')\n            print(\'Reusing verified ZIP:\',result[\'submission_zip\'],flush=True)\n        else:\n            output=run/\'test_predictions.json\';start=time.perf_counter()\n            launch(path,\'generate\',world,split=\'test\',output=output)\n            train=read_json(done)\n            result={**pack_prediction(project,name,output,cfg[\'cache\']),\'method\':cfg[\'method\'],\'architecture\':cfg[\'architecture\'],\n                    \'training_minutes\':round(train[\'training_seconds\']/60,2),\'export_minutes\':round((time.perf_counter()-start)/60,2),\n                    \'best_epoch\':train[\'best_epoch\'],\'dev_token_loss\':train[\'best_dev_loss\'],\'parameters\':train[\'parameters\']}\n            atomic_json(receipt,{\'signature\':signature,\'zip_sha256\':file_hash(result[\'submission_zip\']),\'result\':result})\n        results.append(result);atomic_json(project/\'results.json\',results)\n        print(f"Saved {name}: {result[\'submission_zip\']}",flush=True)\n    hashes={r[\'prediction_sha256\'] for r in results}\n    if len(hashes)<len(results):print(\'WARNING: identical prediction sets detected; they are not distinct submissions.\')\n    print(f\'{len(results)} ZIPs, {len(hashes)} distinct prediction sets. This invocation: {(time.perf_counter()-started)/60:.1f} min (excluding preparation).\')\n    return results\n\ndef validate_dev(project,configs,results,world,cases,metric_device):\n    if cases<=0:return []\n    import importlib.util\n    if any(importlib.util.find_spec(name) is None for name in (\'bert_score\',\'rouge_score\')):\n        subprocess.check_call([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'bert-score\',\'rouge-score\'])\n    cache=torch.load(read_json(next(iter(configs.values())))[\'cache\'],map_location=\'cpu\',weights_only=False)\n    truth={r[\'id\']:r[\'summary\'] for r in cache[\'rows\'] if r[\'split\']==\'dev\'};del cache\n    from s6_metrics import score,release\n    generated={}\n    for result in results:\n        name=result[\'name\'];cfg=read_json(configs[name]);run=Path(cfg[\'run\'])\n        path=run/f\'dev_{cases}_predictions.json\'\n        if not path.exists():launch(configs[name],\'generate\',world,split=\'dev\',output=path,limit=cases)\n        generated[name]=read_json(path)\n    ranked=[]\n    try:\n        for result in results:\n            data=generated[result[\'name\']]\n            metrics=score([x[\'summary\'] for x in data],[truth[x[\'id\']] for x in data],metric_device)\n            ranked.append({**result,\'heldout_probe\':metrics})\n    finally:release()\n    ranked.sort(key=lambda r:r[\'heldout_probe\'][\'bertscore_f1\'],reverse=True)\n    atomic_json(Path(project)/\'dev_probe_leaderboard.json\',ranked)\n    for r in ranked:print(r[\'name\'],r[\'heldout_probe\'])\n    return ranked\n\ndef bundle_submissions(project,results):\n    project=Path(project);out=project/\'phase6_fast_four_submissions.zip\'\n    with zipfile.ZipFile(out,\'w\',zipfile.ZIP_STORED) as z:\n        for result in results:\n            p=Path(result[\'submission_zip\'])\n            z.write(p,p.name)\n    return out\n'
(CODE / 'fast_workflow.py').write_text(module_source, encoding="utf-8")
print("Wrote fast_workflow.py")


## 3 · Audit data and build the shared token cache

The held-out split is formed from official training data after removing document copies and test overlap. Malformed train rows are recorded in the audit; a missing test document stops export. Supplied test reference fields are ignored.


In [ ]:
import importlib
import json
import time
importlib.invalidate_caches()
from fast_data import prepare_fast
from fast_workflow import configurations, run_candidates, validate_dev, bundle_submissions
from s6_data import FILES
from s6_workflow import find_phase1, obtain_data

started=time.perf_counter()
if PHASE1_ROOT is not None and not PHASE1_ROOT.is_dir():
    print('Default Phase 1 path not mounted; trying automatic discovery.')
    PHASE1_ROOT=None
PHASE1_ROOT=find_phase1(PHASE1_ROOT)
previous=Path('/kaggle/working/capstone_phase6_v1/data')
if DATA_ROOT is None and all((previous / f).is_file() for f in FILES):
    DATA_ROOT=previous
data_root=obtain_data(PROJECT,DATA_ROOT,DATA_REVISION)
opts=dict(seed=781, source_tokens=SOURCE_TOKENS, target_tokens=TARGET_TOKENS,
          dev_fraction=.08, dev_max=96, train_pool=TRAIN_POOL, selector_fit_docs=1200,
          selector_shortlist=96, special_ids={}, epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH,
          micro_batch=MICRO_BATCH, accumulation=ACCUMULATION, generation_batch=GENERATION_BATCH,
          max_train_minutes=MAX_TRAIN_MINUTES, dev_loss_cases=32, kd_temperature=2.,
          kd_weight=.5, kd_every=3, checkpointing=CHECKPOINTING,
          warm_checkpoint=str(OLD_MODEL_04_WEIGHTS) if OLD_MODEL_04_WEIGHTS else None,
          warm_cache=str(OLD_MODEL_04_CACHE) if OLD_MODEL_04_CACHE else None)
cache_path, audit=prepare_fast(data_root,PHASE1_ROOT/'tokenizer.json',PROJECT,opts)
cfgs=configurations(PROJECT,PHASE1_ROOT,cache_path,opts,RUN_MODELS)
keys=['counts','source_limit','target_limit','training_summary_tokens_p50_p90_p95',
      'train_references_clipped_fraction','selector_fit_documents']
print(json.dumps({k:audit[k] for k in keys},indent=2))
print(f'Preparation: {(time.perf_counter()-started)/60:.2f} min.')
print('Audit:',PROJECT/'cache'/'audit.json')


## 4 · Train and export

Each model's ZIP is written immediately after its own test generation and ID check. Rerunning this cell reuses completed work. Model 06 needs Model 04's trained checkpoint; Model 04 is automatically scheduled first if you select only Model 06.


In [ ]:
from IPython.display import FileLink, display
results=run_candidates(PROJECT,cfgs,RUN_MODELS,WORLD)
for row in results:
    print(f"{row['name']}: {row['training_minutes']:.2f} min training; "
          f"{row['export_minutes']:.2f} min test export; {row['test_documents']} test cases")
    display(FileLink(row['submission_zip']))
bundle=bundle_submissions(PROJECT,results)
display(FileLink(str(bundle)))
print('Use the individual ZIPs for evaluation. Combined download:',bundle)


## 5 · Optional final dev comparison

This evaluates one common held-out probe per candidate **after** the test ZIPs are safe. BERTScore may need to download its metric model the first time. That model is used only for scoring; the four generators use only your Phase 1 model weights and official Phase 6 training examples. Development results are estimates, not competition scores.


In [ ]:
if EVAL_DEV_CASES:
    try:
        ranked=validate_dev(PROJECT,cfgs,results,WORLD,EVAL_DEV_CASES,'cuda:0')
        if ranked:
            print('Best on the held-out BERTScore probe:',ranked[0]['name'])
            display(FileLink(str(PROJECT/'dev_probe_leaderboard.json')))
    except Exception as error:
        print(f'Optional dev scoring did not finish: {type(error).__name__}: {error}')
        print('The four test ZIPs from the previous cell remain available.')
else:
    print('Optional dev scoring disabled.')
